# 3D Gaussian Splatting (splatfacto) → SAM2-маска объекта → STL — чистый pipeline

Ядро Colab (Python 3.13 / torch 2.11) несовместимо с nerfstudio, поэтому nerfstudio ставится в отдельный venv `/content/nsenv` (Python 3.10), команды `ns-*` вызываются оттуда. Сегментация (Grounding DINO + SAM2) — прямо в ядре, ей нужен свежий torch.

`/content` эфемерный. После пересоздания рантайма — просто выполни все ячейки сверху вниз: готовые шаги сами пропустятся.

**Порядок:** GPU → setup venv (~10 мин, один раз) → датасет truck + фикс интринсик → сегментация SAM2 (~маски по тексту) → обучение с масками (~20 мин) → экспорт гауссиан → Poisson-mesh + DBSCAN → STL.

## 1. GPU и ядро

In [ ]:
import sys
print('kernel:', sys.version.split()[0])
!nvidia-smi -L

## 2. Setup venv (Python 3.10 + torch 2.1.2 + gsplat 1.4.0 прекомпилированный + nerfstudio 1.1.5)

Одна ячейка, ~10 мин. Если venv уже есть — быстрая проверка и выход.

Ключевое: Python **3.10** (готовые колёса gsplat есть только под cp310), gsplat ставится прекомпилированным колесом `+pt21cu118` — никакой компиляции (иначе OOM-kill на RAM Colab). `setuptools<70` — из-за `pkg_resources.packaging`. `numpy==1.26.4` — иначе ломается импорт torch.

In [ ]:
%%bash
set -e
NSPY=/content/nsenv/bin/python

if [ -x /content/nsenv/bin/ns-train ]; then
  echo '=== venv уже собран, проверка ==='
  $NSPY -c "import numpy,torch,gsplat; from nerfstudio.models.splatfacto import SplatfactoModel; print('numpy',numpy.__version__,'| torch',torch.__version__,'| gsplat',gsplat.__version__,'| cuda',torch.cuda.is_available(),'| splatfacto OK')"
  exit 0
fi

pip install -q uv
uv python install 3.10
uv venv --python 3.10 /content/nsenv

uv pip install --python $NSPY "torch==2.1.2" "torchvision==0.16.2" --index-url https://download.pytorch.org/whl/cu118
uv pip install --python $NSPY "setuptools<70" wheel
uv pip install --python $NSPY "gsplat==1.4.0+pt21cu118" --index-url https://docs.gsplat.studio/whl/pt21cu118 --extra-index-url https://pypi.org/simple --index-strategy unsafe-best-match
uv pip install --python $NSPY "nerfstudio==1.1.5" open3d trimesh
uv pip install --python $NSPY "numpy==1.26.4"

echo '=== проверка ==='
$NSPY -c "import numpy,torch,gsplat; from gsplat.cuda._backend import _C; from nerfstudio.models.splatfacto import SplatfactoModel; print('numpy',numpy.__version__,'| torch',torch.__version__,'| gsplat',gsplat.__version__,'| cuda',torch.cuda.is_available(),'| splatfacto OK')"
/content/nsenv/bin/ns-train --help >/dev/null 2>&1 && echo 'ns-train: OK' || echo 'ns-train: FAIL'

## 3. Датасет truck + фикс cameras.bin

Картинки в `images/` уже уменьшены (979×546), а `cameras.bin` хранит интринсики полного размера (1957×1091) → масштабируем fx/fy/cx/cy. Оригинал → `cameras.bin.orig`.

In [ ]:
import os, subprocess

D = '/content/tandt/truck'
if not os.path.isdir(f'{D}/images'):
    os.system('wget -q --show-progress https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/datasets/input/tandt_db.zip -O /content/tandt_db.zip')
    os.system('cd /content && unzip -q -o tandt_db.zip')
print('кадров:', len(os.listdir(f'{D}/images')))
print('COLMAP:', sorted(os.listdir(f'{D}/sparse/0')))

if not os.path.exists(f'{D}/sparse/0/cameras.bin.orig'):
    code = r'''
import glob, shutil
from PIL import Image
from nerfstudio.data.utils.colmap_parsing_utils import read_cameras_binary, write_cameras_binary, Camera
sp, im = "/content/tandt/truck/sparse/0", "/content/tandt/truck/images"
cams = read_cameras_binary(f"{sp}/cameras.bin")
w, h = Image.open(sorted(glob.glob(im + "/*"))[0]).size
new = {}
for k, c in cams.items():
    sx, sy = w / c.width, h / c.height
    p = list(c.params)
    if c.model == "PINHOLE":
        p = [p[0]*sx, p[1]*sy, p[2]*sx, p[3]*sy]
    elif c.model in ("SIMPLE_PINHOLE", "SIMPLE_RADIAL"):
        p[0] *= sx; p[1] *= sx; p[2] *= sy
    else:
        raise SystemExit("модель %s не поддержана" % c.model)
    new[k] = Camera(id=c.id, model=c.model, width=w, height=h, params=p)
shutil.copy(f"{sp}/cameras.bin", f"{sp}/cameras.bin.orig")
write_cameras_binary(new, f"{sp}/cameras.bin")
[print("cameras.bin →", v.width, v.height, [round(x, 2) for x in v.params]) for v in read_cameras_binary(f"{sp}/cameras.bin").values()]
'''
    r = subprocess.run(['/content/nsenv/bin/python', '-c', code], capture_output=True, text=True)
    print(r.stdout or r.stderr[-2000:])
else:
    print('cameras.bin уже пофикшен')

## 4. Сегментация: текст → маска (Grounding DINO + SAM2)

Маска объекта по текстовому промпту (`OBJECT_PROMPT`) строится ДО обучения — фоновые гауссианы никогда не учатся. Это меняет **SAM v1 → SAM2** (`facebook/sam2-hiera-large`, HF `transformers` `Sam2Model`/`Sam2Processor`) — Grounding DINO не трогаем. Работает в ЯДРЕ Colab (Python 3.13/torch 2.11), не в venv.</cell id="cell-6">

In [ ]:
!pip install -q -U transformers
import torch, numpy as np
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, Sam2Model, Sam2Processor

dev = "cuda"
OBJECT_PROMPT = "a truck."  # текстовое описание объекта пользователя — заменить под реальный сценарий
GD, SAM = "IDEA-Research/grounding-dino-base", "facebook/sam2-hiera-large"
gd_proc = AutoProcessor.from_pretrained(GD)
gd_model = AutoModelForZeroShotObjectDetection.from_pretrained(GD).to(dev).eval()
sam_proc = Sam2Processor.from_pretrained(SAM)
sam_model = Sam2Model.from_pretrained(SAM).to(dev).eval()
print("модели ок (SAM2)")

def gdino_boxes(img, text=OBJECT_PROMPT, thr=0.3):
    inp = gd_proc(images=img, text=text, return_tensors="pt").to(dev)
    with torch.no_grad():
        out = gd_model(**inp)
    tgt = [img.size[::-1]]
    try:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    except TypeError:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, box_threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    return r["boxes"].cpu().numpy(), r["scores"].cpu().numpy()

In [ ]:
import os
from tqdm import tqdm

IMG_DIR  = f'{D}/images'
MASK_DIR = f'{D}/masks'
PREV_DIR = f'{D}/_mask_preview'
os.makedirs(MASK_DIR, exist_ok=True); os.makedirs(PREV_DIR, exist_ok=True)

_pp = getattr(sam_proc, "post_process_masks", None) or sam_proc.image_processor.post_process_masks

def sam_union(img, boxes):
    inp = sam_proc(img, input_boxes=[[list(map(float, b)) for b in boxes]], return_tensors="pt").to(dev)
    with torch.no_grad():
        out = sam_model(**inp, multimask_output=True)
    masks = _pp(out.pred_masks.cpu(), inp["original_sizes"].cpu())[0]
    m = masks.numpy()
    iou = out.iou_scores.cpu().numpy()[0]
    sel = np.stack([m[i, iou[i].argmax()] for i in range(m.shape[0])])
    return sel.any(0)

names = sorted(os.listdir(IMG_DIR))
no_box = []
if os.path.isdir(MASK_DIR) and len(os.listdir(MASK_DIR)) == len(names):
    print('маски уже есть:', len(names))
else:
    for i, n in enumerate(tqdm(names)):
        img = Image.open(f"{IMG_DIR}/{n}").convert("RGB")
        W, H = img.size
        b, s = gdino_boxes(img, OBJECT_PROMPT, 0.3)
        if len(b) == 0:
            b, s = gdino_boxes(img, "объект. object. vehicle.", 0.22)
        keep = [bb for bb in b if 0.02 < ((bb[2]-bb[0])*(bb[3]-bb[1]))/(W*H) < 0.99]
        if not keep:
            no_box.append(n)
            mask = np.ones((H, W), bool)
        else:
            mask = sam_union(img, keep)
        Image.fromarray((mask*255).astype(np.uint8)).save(f"{MASK_DIR}/{os.path.splitext(n)[0]}.png")
        if i % 42 == 0:
            ov = np.array(img); ov[~mask] = (ov[~mask]*0.2).astype(np.uint8)
            Image.fromarray(ov).save(f"{PREV_DIR}/{n}")
    print("масок:", len(os.listdir(MASK_DIR)), "| без бокса:", len(no_box), no_box[:10])

## 5. Обучение splatfacto с масками (только объект, без фона)

~20 мин на T4. Маски из шага 4 передаются как `--masks-path masks` — фон никогда не учится. Чекпойнты каждые 2000 шагов в `outputs_masked/truck/splatfacto/<ts>/nerfstudio_models/`.

Если `CUDA out of memory` — поставь `--downscale-factor 2` и `--pipeline.model.num-downscales 2`.

In [ ]:
!yes | TORCHDYNAMO_DISABLE=1 /content/nsenv/bin/ns-train splatfacto --data /content/tandt/truck --max-num-iterations 7000 --steps-per-save 2000 --vis tensorboard --viewer.quit-on-train-completion True --output-dir outputs_masked --pipeline.model.cull-alpha-thresh 0.1 colmap --images-path images --colmap-path sparse/0 --masks-path masks --downscale-factor 1

## 6. Путь к конфигу обученной модели

In [ ]:
import glob, os
cfgs = sorted(glob.glob('outputs_masked/truck/splatfacto/*/config.yml'), key=os.path.getmtime)
assert cfgs, 'config.yml нет — обучение не завершилось'
CONFIG = cfgs[-1]
print('CONFIG =', CONFIG)

## 7. Экспорт гауссиан → .ply

In [ ]:
!/content/nsenv/bin/ns-export gaussian-splat --load-config {CONFIG} --output-dir exports/truck_masked
!ls -la exports/truck_masked

## 8. Poisson-mesh → STL (только объект)

Облако гауссиан (уже без фона благодаря маске) → чистка выбросов → DBSCAN, оставляем крупнейший кластер → нормали → Screened Poisson → STL. Низ объекта, скорее всего, не отснят — меш может быть не watertight (руками в Blender/Meshmixer при необходимости печати).

In [ ]:
script = r'''
import glob
import numpy as np
import open3d as o3d
import trimesh

ply = sorted(glob.glob("exports/truck_masked/*.ply"))[-1]
print("PLY:", ply)
pcd = o3d.io.read_point_cloud(ply)
print("Punkte:", len(pcd.points))
pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
print("nach Bereinigung:", len(pcd.points))

nn = o3d.geometry.KDTreeFlann(pcd)
step = max(1, len(pcd.points) // 2000)
d = []
for i in range(0, len(pcd.points), step):
    _, idx, dist2 = nn.search_knn_vector_3d(pcd.points[i], 2)
    d.append(np.sqrt(dist2[1]))
eps = 5 * float(np.median(d))
labels = np.array(pcd.cluster_dbscan(eps=eps, min_points=10))
if labels.max() >= 0:
    biggest = np.bincount(labels[labels >= 0]).argmax()
    pcd = pcd.select_by_index(np.where(labels == biggest)[0])
print("nach DBSCAN (groesster Cluster):", len(pcd.points))

pcd.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
pcd.orient_normals_consistent_tangent_plane(30)
mesh, dens = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=10)
dens = np.asarray(dens)
mesh.remove_vertices_by_mask(dens <= np.quantile(dens, 0.1))
mesh.remove_degenerate_triangles(); mesh.remove_duplicated_vertices()
mesh.remove_duplicated_triangles(); mesh.remove_non_manifold_edges()
mesh.compute_triangle_normals()
out = "exports/truck_masked/truck_object.stl"
o3d.io.write_triangle_mesh(out, mesh)

m = trimesh.load(out)
m.fill_holes(); m.fix_normals(); m.process(validate=True)
m.export(out)
print("STL:", out, "| Ecken:", len(m.vertices), "| Dreiecke:", len(m.faces))
print("watertight:", m.is_watertight, "| bbox:", [round(float(x), 3) for x in m.bounding_box.extents])
'''
open('/content/mesh_to_stl.py', 'w').write(script)
import subprocess
r = subprocess.run(['/content/nsenv/bin/python', '/content/mesh_to_stl.py'], capture_output=True, text=True)
print(r.stdout)
if r.returncode:
    print('STDERR:\n', r.stderr[-3000:])

## Результат

`exports/truck_masked/truck_object.stl` — только объект (фон срезан маской SAM2+DINO). Взять через панель файлов VS Code (`content/exports/truck_masked/` → правый клик → Download) или скопировать на Drive.

In [ ]:
!apt-get -qq install -y colmap
!colmap -h | head -5


In [ ]:
import os, shutil, random

SRC = f'{D}/images'
RAW = '/content/sparse_test/raw'
os.makedirs(RAW, exist_ok=True)

names = sorted(os.listdir(SRC))
random.seed(0)
subset = sorted(random.sample(names, 20))
for n in subset:
    shutil.copy(f'{SRC}/{n}', f'{RAW}/{n}')

print('скопировано:', len(os.listdir(RAW)))


In [ ]:
import os, shutil, random

SRC = '/content/tandt/truck/images'
RAW = '/content/sparse_test/raw'
os.makedirs(RAW, exist_ok=True)

names = sorted(os.listdir(SRC))
random.seed(0)
subset = sorted(random.sample(names, 20))
for n in subset:
    shutil.copy(f'{SRC}/{n}', f'{RAW}/{n}')

print('скопировано:', len(os.listdir(RAW)))


In [ ]:
!nvidia-smi -L


In [ ]:
img_test = Image.open(f"{IMG_DIR}/{names[0]}").convert("RGB")
inp_test = sam_proc(img_test, input_boxes=[[[0,0,100,100]]], return_tensors="pt").to(dev)
print(list(inp_test.keys()))

import inspect
print(inspect.signature(_pp))


In [ ]:
script = r'''
import glob
import numpy as np
import open3d as o3d
import trimesh

ply = sorted(glob.glob("exports/truck_masked/*.ply"))[-1]
print("PLY:", ply)
pcd = o3d.io.read_point_cloud(ply)
print("Punkte:", len(pcd.points))
pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
print("nach Bereinigung:", len(pcd.points))

nn = o3d.geometry.KDTreeFlann(pcd)
step = max(1, len(pcd.points) // 2000)
d = []
for i in range(0, len(pcd.points), step):
    _, idx, dist2 = nn.search_knn_vector_3d(pcd.points[i], 2)
    d.append(np.sqrt(dist2[1]))
eps = 5 * float(np.median(d))
labels = np.array(pcd.cluster_dbscan(eps=eps, min_points=10))
if labels.max() >= 0:
    biggest = np.bincount(labels[labels >= 0]).argmax()
    pcd = pcd.select_by_index(np.where(labels == biggest)[0])
print("nach DBSCAN (groesster Cluster):", len(pcd.points))

pcd.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
pcd.orient_normals_consistent_tangent_plane(30)
mesh, dens = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=10)
dens = np.asarray(dens)
mesh.remove_vertices_by_mask(dens <= np.quantile(dens, 0.1))
mesh.remove_degenerate_triangles(); mesh.remove_duplicated_vertices()
mesh.remove_duplicated_triangles(); mesh.remove_non_manifold_edges()
mesh.compute_triangle_normals()
out = "exports/truck_masked/truck_object.stl"
o3d.io.write_triangle_mesh(out, mesh)

m = trimesh.load(out)
m.fill_holes(); m.fix_normals(); m.process(validate=True)
m.export(out)
print("STL:", out, "| Ecken:", len(m.vertices), "| Dreiecke:", len(m.faces))
print("watertight:", m.is_watertight, "| bbox:", [round(float(x), 3) for x in m.bounding_box.extents])
'''
open('/content/mesh_to_stl.py', 'w').write(script)
import subprocess
r = subprocess.run(['/content/nsenv/bin/python', '/content/mesh_to_stl.py'], capture_output=True, text=True)
print(r.stdout)
if r.returncode:
    print('STDERR:\n', r.stderr[-3000:])


In [ ]:
!/content/nsenv/bin/ns-process-data images \
  --data /content/sparse_test/raw \
  --output-dir /content/sparse_test/processed \
  --matching-method exhaustive


In [ ]:
!rm -rf /content/sparse_test/processed
!QT_QPA_PLATFORM=offscreen /content/nsenv/bin/ns-process-data images \
  --data /content/sparse_test/raw \
  --output-dir /content/sparse_test/processed \
  --matching-method exhaustive


In [ ]:
import os, shutil, subprocess

N = 12
SRC = "/content/tandt/truck"
DST = f"/content/tandt/truck_sparse{N}"
os.makedirs(f"{DST}/images", exist_ok=True)
os.makedirs(f"{DST}/sparse/0", exist_ok=True)

names = sorted(os.listdir(f"{SRC}/images"))
step = len(names) / N
chosen = [names[int(i * step)] for i in range(N)]
print("выбрано кадров:", len(chosen))
print(chosen)

for n in chosen:
    shutil.copy(f"{SRC}/images/{n}", f"{DST}/images/{n}")
shutil.copy(f"{SRC}/sparse/0/cameras.bin", f"{DST}/sparse/0/cameras.bin")
shutil.copy(f"{SRC}/sparse/0/project.ini", f"{DST}/sparse/0/project.ini")

code = f'''
import numpy as np
from nerfstudio.data.utils.colmap_parsing_utils import (
    read_images_binary, write_images_binary,
    read_points3D_binary, write_points3D_binary)

chosen = {chosen!r}
imgs = read_images_binary("{SRC}/sparse/0/images.bin")
kept = {{k: v for k, v in imgs.items() if v.name in chosen}}
print("оставлено камер:", len(kept), "из", len(imgs))
write_images_binary(kept, "{DST}/sparse/0/images.bin")

kept_ids = set(kept.keys())
pts = read_points3D_binary("{SRC}/sparse/0/points3D.bin")
pts_kept = {{}}
for pid, p in pts.items():
    mask = np.array([i for i, imid in enumerate(p.image_ids) if imid in kept_ids], dtype=int)
    if mask.size == 0:
        continue
    pts_kept[pid] = p._replace(image_ids=p.image_ids[mask], point2D_idxs=p.point2D_idxs[mask])
print("оставлено 3D-точек:", len(pts_kept), "из", len(pts))
write_points3D_binary(pts_kept, "{DST}/sparse/0/points3D.bin")
'''
r = subprocess.run(['/content/nsenv/bin/python', '-c', code], capture_output=True, text=True)
print(r.stdout)
if r.returncode:
    print("STDERR:\n", r.stderr[-3000:])


In [ ]:
!/content/nsenv/bin/ns-process-data images \
  --data /content/sparse_test/raw \
  --output-dir /content/sparse_test/processed \
  --matching-method exhaustive


In [ ]:
!rm -rf /content/sparse_test/processed
!QT_QPA_PLATFORM=offscreen /content/nsenv/bin/ns-process-data images \
  --data /content/sparse_test/raw \
  --output-dir /content/sparse_test/processed \
  --matching-method exhaustive


In [ ]:
!rm -rf /content/sparse_test/processed
!QT_QPA_PLATFORM=offscreen /content/nsenv/bin/ns-process-data images \
  --data /content/sparse_test/raw \
  --output-dir /content/sparse_test/processed \
  --matching-method exhaustive \
  --no-gpu


In [ ]:
!rm -rf /content/sparse_test/processed
!QT_QPA_PLATFORM=offscreen /content/nsenv/bin/ns-process-data images \
  --data /content/sparse_test/raw \
  --output-dir /content/sparse_test/processed \
  --matching-method exhaustive \
  --no-gpu

In [ ]:
!rm -rf /content/sparse_test/processed
!QT_QPA_PLATFORM=offscreen /content/nsenv/bin/ns-process-data images \
  --data /content/sparse_test/raw \
  --output-dir /content/sparse_test/processed \
  --matching-method exhaustive \
  --no-gpu


In [ ]:
!ls /content/nsenv/bin/ 2>&1 | head -3


In [ ]:
import os
os.makedirs('/content/drive/MyDrive/nerf_cache', exist_ok=True)
CACHE = '/content/drive/MyDrive/nerf_cache/content_cache.tar.gz'
if not os.path.exists(CACHE):
    print('Архивирую и заливаю на Drive (может занять пару минут)...')
    os.system(f'tar -czf {CACHE} /content/nsenv /content/tandt')
    print('Готово, размер:', round(os.path.getsize(CACHE)/1e9, 2), 'GB')
else:
    print('Кэш уже есть на Drive.')


In [ ]:
import os
print('venv:', os.path.exists('/content/nsenv/bin/ns-train'))
print('датасет:', os.path.exists('/content/tandt/truck/images'))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CACHE = '/content/drive/MyDrive/nerf_cache/content_cache.tar.gz'
if os.path.exists(CACHE):
    print('Кэш найден, восстанавливаю...')
    os.system(f'tar -xzf {CACHE} -C /')
    print('venv:', os.path.exists('/content/nsenv/bin/ns-train'), '| маски:', os.path.exists('/content/tandt/truck/masks'))
else:
    print('Кэша нет!')


In [ ]:
import os
d = '/content/drive/MyDrive/nerf_cache'
print(os.path.exists(d))
print(os.listdir(d) if os.path.exists(d) else 'папки нет')
print(os.listdir('/content/drive/MyDrive'))


In [ ]:
%%bash
set -e
NSPY=/content/nsenv/bin/python

if [ -x /content/nsenv/bin/ns-train ]; then
  echo '=== venv bereits eingerichtet ==='
  exit 0
fi

pip install -q uv
uv python install 3.10
uv venv --python 3.10 /content/nsenv

uv pip install --python $NSPY "torch==2.1.2" "torchvision==0.16.2" --index-url https://download.pytorch.org/whl/cu118
uv pip install --python $NSPY "setuptools<70" wheel
uv pip install --python $NSPY "gsplat==1.4.0+pt21cu118" --index-url https://docs.gsplat.studio/whl/pt21cu118 --extra-index-url https://pypi.org/simple --index-strategy unsafe-best-match
uv pip install --python $NSPY "nerfstudio==1.1.5" open3d trimesh
uv pip install --python $NSPY "numpy==1.26.4"

$NSPY -c "import numpy,torch,gsplat; from nerfstudio.models.splatfacto import SplatfactoModel; print('numpy',numpy.__version__,'| torch',torch.__version__,'| gsplat',gsplat.__version__,'| cuda',torch.cuda.is_available(),'| splatfacto OK')"


In [ ]:
import os

D = '/content/tandt/truck'
if not os.path.isdir(f'{D}/images'):
    os.system('wget -q --show-progress https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/datasets/input/tandt_db.zip -O /content/tandt_db.zip')
    os.system('cd /content && unzip -q -o tandt_db.zip')
print('кадров:', len(os.listdir(f'{D}/images')))


In [ ]:
!apt-get -qq install -y colmap


In [ ]:
!apt-get -qq update
!apt-get -qq install -y colmap


In [ ]:
!colmap -h | head -3


In [ ]:
import os, shutil, random

SRC = '/content/tandt/truck/images'
RAW = '/content/sparse_test30/raw'
os.makedirs(RAW, exist_ok=True)

names = sorted(os.listdir(SRC))
random.seed(0)
subset = sorted(random.sample(names, 30))
for n in subset:
    shutil.copy(f'{SRC}/{n}', f'{RAW}/{n}')

print('скопировано:', len(os.listdir(RAW)))


In [ ]:
!QT_QPA_PLATFORM=offscreen /content/nsenv/bin/ns-process-data images \
  --data /content/sparse_test30/raw \
  --output-dir /content/sparse_test30/processed \
  --matching-method exhaustive \
  --no-gpu


In [ ]:
import os
print('venv:', os.path.exists('/content/nsenv/bin/ns-train'))
print('sparse20:', os.path.exists('/content/sparse_test/processed/transforms.json'))
print('sparse30:', os.path.exists('/content/sparse_test30/processed/transforms.json'))
print('полный STL:', os.path.exists('/content/exports/truck_masked/truck_object.stl'))


In [ ]:
import os

def check(p):
    return os.path.exists(p)

print('colmap:', os.system('which colmap') == 0)
print('sparse20 raw:', check('/content/sparse_test/raw'), len(os.listdir('/content/sparse_test/raw')) if check('/content/sparse_test/raw') else 0)
print('sparse30 processed images:', len(os.listdir('/content/sparse_test30/processed/images')) if check('/content/sparse_test30/processed/images') else 'нет')
print('full ply:', check('/content/exports/truck_masked/splat.ply'))
print('full masks:', len(os.listdir('/content/tandt/truck/masks')) if check('/content/tandt/truck/masks') else 'нет')
print('training outputs:', os.listdir('/content/outputs_masked/truck/splatfacto') if check('/content/outputs_masked/truck/splatfacto') else 'нет')


In [ ]:
import os

D = '/content/tandt/truck'
if not os.path.isdir(f'{D}/images'):
    os.system('wget -q --show-progress https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/datasets/input/tandt_db.zip -O /content/tandt_db.zip')
    os.system('cd /content && unzip -q -o tandt_db.zip')
print('кадров:', len(os.listdir(f'{D}/images')))


In [ ]:
!pip install -q -U transformers
import torch, numpy as np
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, Sam2Model, Sam2Processor

dev = "cuda"
OBJECT_PROMPT = "a truck."
GD, SAM = "IDEA-Research/grounding-dino-base", "facebook/sam2-hiera-large"
gd_proc = AutoProcessor.from_pretrained(GD)
gd_model = AutoModelForZeroShotObjectDetection.from_pretrained(GD).to(dev).eval()
sam_proc = Sam2Processor.from_pretrained(SAM)
sam_model = Sam2Model.from_pretrained(SAM).to(dev).eval()
print("модели ок (SAM2)")

def gdino_boxes(img, text=OBJECT_PROMPT, thr=0.3):
    inp = gd_proc(images=img, text=text, return_tensors="pt").to(dev)
    with torch.no_grad():
        out = gd_model(**inp)
    tgt = [img.size[::-1]]
    try:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    except TypeError:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, box_threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    return r["boxes"].cpu().numpy(), r["scores"].cpu().numpy()


In [ ]:
!pip install -q -U transformers
import torch, numpy as np
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, Sam2Model, Sam2Processor

dev = "cuda"
OBJECT_PROMPT = "a truck."
GD, SAM = "IDEA-Research/grounding-dino-base", "facebook/sam2-hiera-large"
gd_proc = AutoProcessor.from_pretrained(GD)
gd_model = AutoModelForZeroShotObjectDetection.from_pretrained(GD).to(dev).eval()
sam_proc = Sam2Processor.from_pretrained(SAM)
sam_model = Sam2Model.from_pretrained(SAM).to(dev).eval()
print("модели ок (SAM2)")

def gdino_boxes(img, text=OBJECT_PROMPT, thr=0.3):
    inp = gd_proc(images=img, text=text, return_tensors="pt").to(dev)
    with torch.no_grad():
        out = gd_model(**inp)
    tgt = [img.size[::-1]]
    try:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    except TypeError:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, box_threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    return r["boxes"].cpu().numpy(), r["scores"].cpu().numpy()


In [ ]:
import os
from tqdm import tqdm

D = '/content/tandt/truck'
IMG_DIR  = f'{D}/images'
MASK_DIR = f'{D}/masks'
PREV_DIR = f'{D}/_mask_preview'
os.makedirs(MASK_DIR, exist_ok=True); os.makedirs(PREV_DIR, exist_ok=True)

_pp = getattr(sam_proc, "post_process_masks", None) or sam_proc.image_processor.post_process_masks

def sam_union(img, boxes):
    inp = sam_proc(img, input_boxes=[[list(map(float, b)) for b in boxes]], return_tensors="pt").to(dev)
    with torch.no_grad():
        out = sam_model(**inp, multimask_output=True)
    masks = _pp(out.pred_masks.cpu(), inp["original_sizes"].cpu())[0]
    m = masks.numpy()
    iou = out.iou_scores.cpu().numpy()[0]
    sel = np.stack([m[i, iou[i].argmax()] for i in range(m.shape[0])])
    return sel.any(0)

names = sorted(os.listdir(IMG_DIR))
no_box = []
for i, n in enumerate(tqdm(names)):
    img = Image.open(f"{IMG_DIR}/{n}").convert("RGB")
    W, H = img.size
    b, s = gdino_boxes(img, OBJECT_PROMPT, 0.3)
    if len(b) == 0:
        b, s = gdino_boxes(img, "object. vehicle. truck.", 0.22)
    keep = [bb for bb in b if 0.02 < ((bb[2]-bb[0])*(bb[3]-bb[1]))/(W*H) < 0.99]
    if not keep:
        no_box.append(n)
        mask = np.ones((H, W), bool)
    else:
        mask = sam_union(img, keep)
    Image.fromarray((mask*255).astype(np.uint8)).save(f"{MASK_DIR}/{os.path.splitext(n)[0]}.png")
print("масок:", len(os.listdir(MASK_DIR)), "| без бокса:", len(no_box))


In [ ]:
import os

D = '/content/tandt/truck'
if not os.path.isdir(f'{D}/images'):
    os.system('wget -q --show-progress https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/datasets/input/tandt_db.zip -O /content/tandt_db.zip')
    os.system('cd /content && unzip -q -o tandt_db.zip')
print('кадров:', len(os.listdir(f'{D}/images')))


In [ ]:
import os
from tqdm import tqdm

D = '/content/tandt/truck'
IMG_DIR  = f'{D}/images'
MASK_DIR = f'{D}/masks'
PREV_DIR = f'{D}/_mask_preview'
os.makedirs(MASK_DIR, exist_ok=True); os.makedirs(PREV_DIR, exist_ok=True)

_pp = getattr(sam_proc, "post_process_masks", None) or sam_proc.image_processor.post_process_masks

def sam_union(img, boxes):
    inp = sam_proc(img, input_boxes=[[list(map(float, b)) for b in boxes]], return_tensors="pt").to(dev)
    with torch.no_grad():
        out = sam_model(**inp, multimask_output=True)
    masks = _pp(out.pred_masks.cpu(), inp["original_sizes"].cpu())[0]
    m = masks.numpy()
    iou = out.iou_scores.cpu().numpy()[0]
    sel = np.stack([m[i, iou[i].argmax()] for i in range(m.shape[0])])
    return sel.any(0)

names = sorted(os.listdir(IMG_DIR))
no_box = []
for i, n in enumerate(tqdm(names)):
    img = Image.open(f"{IMG_DIR}/{n}").convert("RGB")
    W, H = img.size
    b, s = gdino_boxes(img, OBJECT_PROMPT, 0.3)
    if len(b) == 0:
        b, s = gdino_boxes(img, "object. vehicle. truck.", 0.22)
    keep = [bb for bb in b if 0.02 < ((bb[2]-bb[0])*(bb[3]-bb[1]))/(W*H) < 0.99]
    if not keep:
        no_box.append(n)
        mask = np.ones((H, W), bool)
    else:
        mask = sam_union(img, keep)
    Image.fromarray((mask*255).astype(np.uint8)).save(f"{MASK_DIR}/{os.path.splitext(n)[0]}.png")
print("масок:", len(os.listdir(MASK_DIR)), "| без бокса:", len(no_box))


In [ ]:
!yes | TORCHDYNAMO_DISABLE=1 /content/nsenv/bin/ns-train splatfacto --data /content/tandt/truck --max-num-iterations 7000 --steps-per-save 2000 --vis tensorboard --viewer.quit-on-train-completion True --output-dir outputs_masked --pipeline.model.cull-alpha-thresh 0.1 colmap --images-path images --colmap-path sparse/0 --masks-path masks --downscale-factor 1


In [ ]:
import os
print('venv:', os.path.exists('/content/nsenv/bin/ns-train'))
print('датасет:', os.path.exists('/content/tandt/truck/images'))
print('маски:', len(os.listdir('/content/tandt/truck/masks')) if os.path.exists('/content/tandt/truck/masks') else 0)


In [ ]:
%%bash
set -e
NSPY=/content/nsenv/bin/python
pip install -q uv
uv python install 3.10
uv venv --python 3.10 /content/nsenv
uv pip install --python $NSPY "torch==2.1.2" "torchvision==0.16.2" --index-url https://download.pytorch.org/whl/cu118
uv pip install --python $NSPY "setuptools<70" wheel
uv pip install --python $NSPY "gsplat==1.4.0+pt21cu118" --index-url https://docs.gsplat.studio/whl/pt21cu118 --extra-index-url https://pypi.org/simple --index-strategy unsafe-best-match
uv pip install --python $NSPY "nerfstudio==1.1.5" open3d trimesh
uv pip install --python $NSPY "numpy==1.26.4"
$NSPY -c "import torch; print('cuda', torch.cuda.is_available())"


In [ ]:
!yes | TORCHDYNAMO_DISABLE=1 /content/nsenv/bin/ns-train splatfacto --data /content/tandt/truck --max-num-iterations 7000 --steps-per-save 2000 --vis tensorboard --viewer.quit-on-train-completion True --output-dir outputs_masked --pipeline.model.cull-alpha-thresh 0.1 colmap --images-path images --colmap-path sparse/0 --masks-path masks --downscale-factor 1


In [ ]:
import subprocess

code = r'''
import glob, shutil
from PIL import Image
from nerfstudio.data.utils.colmap_parsing_utils import read_cameras_binary, write_cameras_binary, Camera
sp, im = "/content/tandt/truck/sparse/0", "/content/tandt/truck/images"
cams = read_cameras_binary(f"{sp}/cameras.bin")
w, h = Image.open(sorted(glob.glob(im + "/*"))[0]).size
new = {}
for k, c in cams.items():
    sx, sy = w / c.width, h / c.height
    p = list(c.params)
    if c.model == "PINHOLE":
        p = [p[0]*sx, p[1]*sy, p[2]*sx, p[3]*sy]
    new[k] = Camera(id=c.id, model=c.model, width=w, height=h, params=p)
shutil.copy(f"{sp}/cameras.bin", f"{sp}/cameras.bin.orig")
write_cameras_binary(new, f"{sp}/cameras.bin")
print("готово:", [v.width for v in read_cameras_binary(f"{sp}/cameras.bin").values()][:1])
'''
r = subprocess.run(['/content/nsenv/bin/python', '-c', code], capture_output=True, text=True)
print(r.stdout or r.stderr[-2000:])

In [ ]:
import os
print('venv:', os.path.exists('/content/nsenv/bin/ns-train'))
print('датасет:', os.path.exists('/content/tandt/truck/images'))
print('маски:', len(os.listdir('/content/tandt/truck/masks')) if os.path.exists('/content/tandt/truck/masks') else 0)


In [ ]:
%%bash
set -e
NSPY=/content/nsenv/bin/python

if [ ! -x /content/nsenv/bin/ns-train ]; then
  pip install -q uv
  uv python install 3.10
  uv venv --python 3.10 /content/nsenv
  uv pip install --python $NSPY "torch==2.1.2" "torchvision==0.16.2" --index-url https://download.pytorch.org/whl/cu118
  uv pip install --python $NSPY "setuptools<70" wheel
  uv pip install --python $NSPY "gsplat==1.4.0+pt21cu118" --index-url https://docs.gsplat.studio/whl/pt21cu118 --extra-index-url https://pypi.org/simple --index-strategy unsafe-best-match
  uv pip install --python $NSPY "nerfstudio==1.1.5" open3d trimesh
  uv pip install --python $NSPY "numpy==1.26.4"
fi

if [ ! -d /content/tandt/truck/images ]; then
  wget -q https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/datasets/input/tandt_db.zip -O /content/tandt_db.zip
  cd /content && unzip -q -o tandt_db.zip
fi

if [ ! -f /content/tandt/truck/sparse/0/cameras.bin.orig ]; then
$NSPY -c "
import glob, shutil
from PIL import Image
from nerfstudio.data.utils.colmap_parsing_utils import read_cameras_binary, write_cameras_binary, Camera
sp, im = '/content/tandt/truck/sparse/0', '/content/tandt/truck/images'
cams = read_cameras_binary(f'{sp}/cameras.bin')
w, h = Image.open(sorted(glob.glob(im + '/*'))[0]).size
new = {}
for k, c in cams.items():
    sx, sy = w / c.width, h / c.height
    p = list(c.params)
    if c.model == 'PINHOLE':
        p = [p[0]*sx, p[1]*sy, p[2]*sx, p[3]*sy]
    new[k] = Camera(id=c.id, model=c.model, width=w, height=h, params=p)
shutil.copy(f'{sp}/cameras.bin', f'{sp}/cameras.bin.orig')
write_cameras_binary(new, f'{sp}/cameras.bin')
"
fi

echo '=== готово ==='
$NSPY -c "import torch; print('cuda', torch.cuda.is_available())"
ls /content/tandt/truck/images | wc -l


In [ ]:
!pip install -q -U transformers
import torch, numpy as np
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, Sam2Model, Sam2Processor

dev = "cuda"
OBJECT_PROMPT = "a truck."
GD, SAM = "IDEA-Research/grounding-dino-base", "facebook/sam2-hiera-large"
gd_proc = AutoProcessor.from_pretrained(GD)
gd_model = AutoModelForZeroShotObjectDetection.from_pretrained(GD).to(dev).eval()
sam_proc = Sam2Processor.from_pretrained(SAM)
sam_model = Sam2Model.from_pretrained(SAM).to(dev).eval()
print("модели ок (SAM2)")

def gdino_boxes(img, text=OBJECT_PROMPT, thr=0.3):
    inp = gd_proc(images=img, text=text, return_tensors="pt").to(dev)
    with torch.no_grad():
        out = gd_model(**inp)
    tgt = [img.size[::-1]]
    try:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    except TypeError:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, box_threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    return r["boxes"].cpu().numpy(), r["scores"].cpu().numpy()


In [ ]:
import os
from tqdm import tqdm

D = '/content/tandt/truck'
IMG_DIR  = f'{D}/images'
MASK_DIR = f'{D}/masks'
os.makedirs(MASK_DIR, exist_ok=True)

_pp = getattr(sam_proc, "post_process_masks", None) or sam_proc.image_processor.post_process_masks

def sam_union(img, boxes):
    inp = sam_proc(img, input_boxes=[[list(map(float, b)) for b in boxes]], return_tensors="pt").to(dev)
    with torch.no_grad():
        out = sam_model(**inp, multimask_output=True)
    masks = _pp(out.pred_masks.cpu(), inp["original_sizes"].cpu())[0]
    m = masks.numpy()
    iou = out.iou_scores.cpu().numpy()[0]
    sel = np.stack([m[i, iou[i].argmax()] for i in range(m.shape[0])])
    return sel.any(0)

names = sorted(os.listdir(IMG_DIR))
no_box = []
for i, n in enumerate(tqdm(names)):
    img = Image.open(f"{IMG_DIR}/{n}").convert("RGB")
    W, H = img.size
    b, s = gdino_boxes(img, OBJECT_PROMPT, 0.3)
    if len(b) == 0:
        b, s = gdino_boxes(img, "object. vehicle. truck.", 0.22)
    keep = [bb for bb in b if 0.02 < ((bb[2]-bb[0])*(bb[3]-bb[1]))/(W*H) < 0.99]
    if not keep:
        no_box.append(n)
        mask = np.ones((H, W), bool)
    else:
        mask = sam_union(img, keep)
    Image.fromarray((mask*255).astype(np.uint8)).save(f"{MASK_DIR}/{os.path.splitext(n)[0]}.png")
print("масок:", len(os.listdir(MASK_DIR)), "| без бокса:", len(no_box))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, time
os.makedirs('/content/drive/MyDrive/nerf_cache', exist_ok=True)
CACHE = '/content/drive/MyDrive/nerf_cache/masks_251.tar.gz'
os.system(f'tar -czf {CACHE} -C /content/tandt/truck masks')
time.sleep(3)
print('размер на Drive:', os.path.getsize(CACHE), 'байт')
print(os.listdir('/content/drive/MyDrive/nerf_cache'))


In [ ]:
!yes | TORCHDYNAMO_DISABLE=1 /content/nsenv/bin/ns-train splatfacto --data /content/tandt/truck --max-num-iterations 7000 --steps-per-save 2000 --vis tensorboard --viewer.quit-on-train-completion True --output-dir outputs_masked --pipeline.model.cull-alpha-thresh 0.1 colmap --images-path images --colmap-path sparse/0 --masks-path masks --downscale-factor 1


In [ ]:
!ls outputs_masked/truck/splatfacto/*/config.yml


In [ ]:
!/content/nsenv/bin/ns-export gaussian-splat --load-config outputs_masked/truck/splatfacto/2026-09-17_130237/config.yml --output-dir exports/truck_masked
!ls -la exports/truck_masked


In [ ]:
import shutil, os
shutil.copy('exports/truck_masked/splat.ply', '/content/drive/MyDrive/nerf_cache/splat_251_ref.ply')
print(os.path.getsize('/content/drive/MyDrive/nerf_cache/splat_251_ref.ply'))


In [ ]:
script = r'''
import glob
import numpy as np
import open3d as o3d
import trimesh

ply = "exports/truck_masked/splat.ply"
pcd = o3d.io.read_point_cloud(ply)
print("Punkte:", len(pcd.points))
pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
print("nach Bereinigung:", len(pcd.points))

nn = o3d.geometry.KDTreeFlann(pcd)
step = max(1, len(pcd.points) // 2000)
d = []
for i in range(0, len(pcd.points), step):
    _, idx, dist2 = nn.search_knn_vector_3d(pcd.points[i], 2)
    d.append(np.sqrt(dist2[1]))
eps = 5 * float(np.median(d))
labels = np.array(pcd.cluster_dbscan(eps=eps, min_points=10))
if labels.max() >= 0:
    biggest = np.bincount(labels[labels >= 0]).argmax()
    pcd = pcd.select_by_index(np.where(labels == biggest)[0])
print("nach DBSCAN:", len(pcd.points))

pcd.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
pcd.orient_normals_consistent_tangent_plane(30)
mesh, dens = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=10)
dens = np.asarray(dens)
mesh.remove_vertices_by_mask(dens <= np.quantile(dens, 0.1))
mesh.remove_degenerate_triangles(); mesh.remove_duplicated_vertices()
mesh.remove_duplicated_triangles(); mesh.remove_non_manifold_edges()
out = "exports/truck_masked/truck_object_ref.stl"
o3d.io.write_triangle_mesh(out, mesh)

m = trimesh.load(out)
m.fill_holes(); m.fix_normals(); m.process(validate=True)
m.export(out)
print("STL:", out, "| Ecken:", len(m.vertices), "| Dreiecke:", len(m.faces))
'''
open('/content/mesh_ref.py', 'w').write(script)
import subprocess
r = subprocess.run(['/content/nsenv/bin/python', '/content/mesh_ref.py'], capture_output=True, text=True)
print(r.stdout)
if r.returncode:
    print('STDERR:\n', r.stderr[-3000:])


In [ ]:
script = r'''
import glob
import numpy as np
import open3d as o3d
import trimesh

ply = "exports/truck_masked/splat.ply"
pcd = o3d.io.read_point_cloud(ply)
print("Punkte:", len(pcd.points))
pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
print("nach Bereinigung:", len(pcd.points))

nn = o3d.geometry.KDTreeFlann(pcd)
step = max(1, len(pcd.points) // 2000)
d = []
for i in range(0, len(pcd.points), step):
    _, idx, dist2 = nn.search_knn_vector_3d(pcd.points[i], 2)
    d.append(np.sqrt(dist2[1]))
eps = 5 * float(np.median(d))
labels = np.array(pcd.cluster_dbscan(eps=eps, min_points=10))
if labels.max() >= 0:
    biggest = np.bincount(labels[labels >= 0]).argmax()
    pcd = pcd.select_by_index(np.where(labels == biggest)[0])
print("nach DBSCAN:", len(pcd.points))

pcd.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
pcd.orient_normals_consistent_tangent_plane(30)
mesh, dens = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=10)
dens = np.asarray(dens)
mesh.remove_vertices_by_mask(dens <= np.quantile(dens, 0.1))
mesh.remove_degenerate_triangles(); mesh.remove_duplicated_vertices()
mesh.remove_duplicated_triangles(); mesh.remove_non_manifold_edges()
mesh.compute_triangle_normals()
out = "exports/truck_masked/truck_object_ref.stl"
o3d.io.write_triangle_mesh(out, mesh)

m = trimesh.load(out)
m.fill_holes(); m.fix_normals(); m.process(validate=True)
m.export(out)
print("STL:", out, "| Ecken:", len(m.vertices), "| Dreiecke:", len(m.faces))
'''
open('/content/mesh_ref.py', 'w').write(script)
import subprocess
r = subprocess.run(['/content/nsenv/bin/python', '/content/mesh_ref.py'], capture_output=True, text=True)
print(r.stdout)
if r.returncode:
    print('STDERR:\n', r.stderr[-3000:])


In [ ]:
import shutil
shutil.copy('exports/truck_masked/truck_object_ref.stl', '/content/drive/MyDrive/nerf_cache/truck_object_ref.stl')
print('сохранено')


In [ ]:
import os
print('sparse20:', os.path.exists('/content/sparse_test/processed/transforms.json'))
print('sparse30:', os.path.exists('/content/sparse_test30/processed/transforms.json'))


In [ ]:
import os
print('colmap:', os.system('which colmap') == 0)


In [ ]:
!apt-get -qq update
!apt-get -qq install -y colmap
!colmap -h | head -2


In [ ]:
import os, shutil, random

SRC = '/content/tandt/truck/images'
names = sorted(os.listdir(SRC))
random.seed(0)

for n_photos, tag in [(20, 'sparse_test'), (30, 'sparse_test30')]:
    RAW = f'/content/{tag}/raw'
    os.makedirs(RAW, exist_ok=True)
    subset = sorted(random.Random(0).sample(names, n_photos))
    for n in subset:
        shutil.copy(f'{SRC}/{n}', f'{RAW}/{n}')
    print(tag, '- скопировано:', len(os.listdir(RAW)))


In [ ]:
!rm -rf /content/sparse_test/processed
!QT_QPA_PLATFORM=offscreen /content/nsenv/bin/ns-process-data images \
  --data /content/sparse_test/raw \
  --output-dir /content/sparse_test/processed \
  --matching-method exhaustive \
  --no-gpu


In [ ]:
!rm -rf /content/sparse_test30/processed
!QT_QPA_PLATFORM=offscreen /content/nsenv/bin/ns-process-data images \
  --data /content/sparse_test30/raw \
  --output-dir /content/sparse_test30/processed \
  --matching-method exhaustive \
  --no-gpu


In [ ]:
import os
os.system('tar -czf /content/drive/MyDrive/nerf_cache/sparse30_processed.tar.gz -C /content/sparse_test30 processed')
print(os.path.getsize('/content/drive/MyDrive/nerf_cache/sparse30_processed.tar.gz'))


In [ ]:
import os
from tqdm import tqdm

IMG_DIR30 = '/content/sparse_test30/processed/images'
MASK_DIR30 = '/content/sparse_test30/processed/masks'
os.makedirs(MASK_DIR30, exist_ok=True)

names30 = sorted(os.listdir(IMG_DIR30))
print('фото в processed:', len(names30))

no_box30 = []
for n in tqdm(names30):
    img = Image.open(f"{IMG_DIR30}/{n}").convert("RGB")
    W, H = img.size
    b, s = gdino_boxes(img, OBJECT_PROMPT, 0.3)
    if len(b) == 0:
        b, s = gdino_boxes(img, "object. vehicle. truck.", 0.22)
    keep = [bb for bb in b if 0.02 < ((bb[2]-bb[0])*(bb[3]-bb[1]))/(W*H) < 0.99]
    if not keep:
        no_box30.append(n)
        mask = np.ones((H, W), bool)
    else:
        mask = sam_union(img, keep)
    Image.fromarray((mask*255).astype(np.uint8)).save(f"{MASK_DIR30}/{os.path.splitext(n)[0]}.png")
print("масок:", len(os.listdir(MASK_DIR30)), "| без бокса:", len(no_box30))


In [ ]:
!yes | TORCHDYNAMO_DISABLE=1 /content/nsenv/bin/ns-train splatfacto \
  --data /content/sparse_test30/processed \
  --max-num-iterations 3000 --steps-per-save 1000 \
  --vis tensorboard --viewer.quit-on-train-completion True \
  --output-dir outputs_sparse30 --pipeline.model.cull-alpha-thresh 0.1 \
  nerfstudio-data --masks-path masks


In [ ]:
!find /content/sparse_test30/processed -maxdepth 3 -type d


In [ ]:
!yes | TORCHDYNAMO_DISABLE=1 /content/nsenv/bin/ns-train splatfacto \
  --data /content/sparse_test30/processed \
  --max-num-iterations 3000 --steps-per-save 1000 \
  --vis tensorboard --viewer.quit-on-train-completion True \
  --output-dir outputs_sparse30 --pipeline.model.cull-alpha-thresh 0.1 \
  colmap --images-path images --colmap-path colmap/sparse/0 --masks-path masks --downscale-factor 1


In [ ]:
!ls outputs_sparse30/images/splatfacto/*/config.yml 2>/dev/null || ls outputs_sparse30/*/splatfacto/*/config.yml


In [ ]:
!/content/nsenv/bin/ns-export gaussian-splat --load-config outputs_sparse30/processed/splatfacto/2026-09-17_133934/config.yml --output-dir exports/sparse30
!ls -la exports/sparse30


In [ ]:
import shutil
shutil.copy('exports/sparse30/splat.ply', '/content/drive/MyDrive/nerf_cache/splat_sparse30.ply')
print('сохранено')


In [ ]:
script = r'''
import numpy as np
import open3d as o3d
import trimesh

ply = "exports/sparse30/splat.ply"
pcd = o3d.io.read_point_cloud(ply)
print("Punkte:", len(pcd.points))
pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
print("nach Bereinigung:", len(pcd.points))

nn = o3d.geometry.KDTreeFlann(pcd)
step = max(1, len(pcd.points) // 2000)
d = []
for i in range(0, len(pcd.points), step):
    _, idx, dist2 = nn.search_knn_vector_3d(pcd.points[i], 2)
    d.append(np.sqrt(dist2[1]))
eps = 5 * float(np.median(d))
labels = np.array(pcd.cluster_dbscan(eps=eps, min_points=10))
if labels.max() >= 0:
    biggest = np.bincount(labels[labels >= 0]).argmax()
    pcd = pcd.select_by_index(np.where(labels == biggest)[0])
print("nach DBSCAN:", len(pcd.points))

pcd.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
pcd.orient_normals_consistent_tangent_plane(30)
mesh, dens = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=9)
dens = np.asarray(dens)
mesh.remove_vertices_by_mask(dens <= np.quantile(dens, 0.1))
mesh.remove_degenerate_triangles(); mesh.remove_duplicated_vertices()
mesh.remove_duplicated_triangles(); mesh.remove_non_manifold_edges()
mesh.compute_triangle_normals()
out = "exports/sparse30/sparse30_object.stl"
o3d.io.write_triangle_mesh(out, mesh)

m = trimesh.load(out)
m.fill_holes(); m.fix_normals(); m.process(validate=True)
m.export(out)
print("STL:", out, "| Ecken:", len(m.vertices), "| Dreiecke:", len(m.faces))
'''
open('/content/mesh_sparse30.py', 'w').write(script)
import subprocess
r = subprocess.run(['/content/nsenv/bin/python', '/content/mesh_sparse30.py'], capture_output=True, text=True)
print(r.stdout)
if r.returncode:
    print('STDERR:\n', r.stderr[-3000:])


In [ ]:
script = r'''
import numpy as np
import open3d as o3d
import trimesh

def load_pts(path, n=30000):
    m = trimesh.load(path)
    pts, _ = trimesh.sample.sample_surface(m, n)
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pts)
    return pcd

def normalize(pcd):
    pts = np.asarray(pcd.points)
    c = pts.mean(0)
    pts = pts - c
    scale = np.linalg.norm(pts, axis=1).max()
    pts = pts / scale
    out = o3d.geometry.PointCloud()
    out.points = o3d.utility.Vector3dVector(pts)
    return out

ref = normalize(load_pts("exports/truck_masked/truck_object_ref.stl"))
sparse = normalize(load_pts("exports/sparse30/sparse30_object.stl"))

# перебираем повороты вокруг вертикальной оси (Z), т.к. масштаб/сдвиг уже сняты нормализацией,
# но угол разворота между двумя независимыми COLMAP-реконструкциями произвольный
best = None
for deg in range(0, 360, 15):
    R = o3d.geometry.get_rotation_matrix_from_axis_angle([0, 0, np.deg2rad(deg)])
    cand = o3d.geometry.PointCloud(sparse)
    cand.rotate(R, center=(0, 0, 0))
    reg = o3d.pipelines.registration.registration_icp(
        cand, ref, 0.05, np.eye(4),
        o3d.pipelines.registration.TransformationEstimationPointToPoint()
    )
    if best is None or reg.inlier_rmse < best[0]:
        best = (reg.inlier_rmse, deg, reg.transformation, R)

rmse, deg, T, R0 = best
print("лучший начальный угол:", deg, "| rmse после ICP:", rmse)

sparse_aligned = o3d.geometry.PointCloud(sparse)
sparse_aligned.rotate(R0, center=(0, 0, 0))
sparse_aligned.transform(T)

# финальный дожим ICP
reg2 = o3d.pipelines.registration.registration_icp(
    sparse_aligned, ref, 0.03, np.eye(4),
    o3d.pipelines.registration.TransformationEstimationPointToPlane()
    if False else o3d.pipelines.registration.TransformationEstimationPointToPoint()
)
sparse_aligned.transform(reg2.transformation)

d1 = np.asarray(sparse_aligned.compute_point_cloud_distance(ref))
d2 = np.asarray(ref.compute_point_cloud_distance(sparse_aligned))
chamfer = d1.mean() + d2.mean()
print(f"Chamfer distance (нормализовано на размер объекта): {chamfer:.4f}")
print(f"  -> sparse30 к ref: {d1.mean():.4f} (медиана {np.median(d1):.4f})")
print(f"  -> ref к sparse30: {d2.mean():.4f} (медиана {np.median(d2):.4f})")
'''
open('/content/chamfer.py', 'w').write(script)
import subprocess
r = subprocess.run(['/content/nsenv/bin/python', '/content/chamfer.py'], capture_output=True, text=True)
print(r.stdout)
if r.returncode:
    print('STDERR:\n', r.stderr[-3000:])


## 9. Реальный фототест: Amazon Berkeley Objects (ABO) — настоящие фото + внешняя CAD-модель

Всё выше (Teil 1-8, sparse-view тесты) — только датасет `truck` (Tanks & Temples), не реальные
фото пользователя. Своих фото снять было негде, поэтому взяла готовый публичный датасет —
**Amazon Berkeley Objects** (CC BY 4.0, (c) Amazon.com, UC Berkeley,
https://amazon-berkeley-objects.s3.amazonaws.com/index.html). Там настоящие фотографии
(не рендеры) реальных товаров на поворотном столе, 72 кадра с шагом 5°, и у части товаров
есть ещё и настоящая CAD-модель (glTF) того же физического объекта — значит можно наконец
посчитать Chamfer-дистанцию против реальной внешней геометрии, а не против self-consistency
proxy из раздела 5 PROJECT_STATUS.md.

Объект — кресло "Ravenna Home Radford Modern Curved" (item_id `B07F2X8K62`, spin_id
`f8022078`), 72 фото + CAD (glb). Прогоняем весь pipeline (COLMAP с нуля → Grounding DINO +
SAM2 маски → masked splatfacto → Poisson → STL) на трёх уровнях: dense (72, референс), N=30,
N=20 — и каждый результат сравниваем Chamfer-дистанцией с настоящим CAD.

Атрибуция при сдаче/публикации обязательна: данные (c) Amazon.com, датасет Amazon Berkeley
Objects, лицензия CC BY 4.0.

In [ ]:
import os, subprocess

ABO_ITEM = "B07F2X8K62"
ABO_SPIN_ID = "f8022078"
ABO_SPIN_PREFIX = ABO_SPIN_ID[:2]
ABO_N_FRAMES = 72
D_ABO = "/content/abo_chair"
os.makedirs(f"{D_ABO}/raw", exist_ok=True)

base = "https://amazon-berkeley-objects.s3.amazonaws.com"
for az in range(ABO_N_FRAMES):
    fn = f"{ABO_SPIN_ID}_{az:02d}.jpg"
    url = f"{base}/spins/original/{ABO_SPIN_PREFIX}/{ABO_SPIN_ID}/{fn}"
    out = f"{D_ABO}/raw/{fn}"
    if not os.path.exists(out):
        subprocess.run(["curl", "-s", "-L", url, "-o", out], check=True)
print("фото скачано:", len(os.listdir(f"{D_ABO}/raw")))

cad_dir = f"{D_ABO}/cad"
os.makedirs(cad_dir, exist_ok=True)
cad_url = f"{base}/3dmodels/original/{ABO_ITEM[-1]}/{ABO_ITEM}.glb"
subprocess.run(["curl", "-s", "-L", cad_url, "-o", f"{cad_dir}/{ABO_ITEM}.glb"], check=True)
print("CAD скачан:", os.listdir(cad_dir))
print("Источник: Amazon Berkeley Objects, CC BY 4.0, (c) Amazon.com")

In [ ]:
import shutil
import numpy as np

def make_subset(n, raw_dir=f"{D_ABO}/raw", out_name=None):
    out_name = out_name or f"raw_N{n}"
    out_dir = f"{D_ABO}/{out_name}"
    if os.path.isdir(out_dir):
        shutil.rmtree(out_dir)
    os.makedirs(out_dir)
    names = sorted(os.listdir(raw_dir))
    idx = sorted(set(np.linspace(0, len(names) - 1, n).round().astype(int).tolist()))
    for i in idx:
        shutil.copy(f"{raw_dir}/{names[i]}", out_dir)
    print(out_name, "->", len(os.listdir(out_dir)), "фото")

make_subset(ABO_N_FRAMES, out_name="raw_dense")  # полная turntable-серия как референс
make_subset(30)
make_subset(20)

In [ ]:
def run_colmap(subset_dir_name):
    raw = f"{D_ABO}/{subset_dir_name}"
    processed = f"{D_ABO}/{subset_dir_name}_processed"
    subprocess.run(f"rm -rf {processed}", shell=True)
    cmd = (
        f"QT_QPA_PLATFORM=offscreen /content/nsenv/bin/ns-process-data images "
        f"--data {raw} --output-dir {processed} --matching-method exhaustive --no-gpu"
    )
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(subset_dir_name, "-> returncode", r.returncode)
    print(r.stdout[-1500:])
    if r.returncode:
        print(r.stderr[-1500:])

for name in ["raw_dense", "raw_N30", "raw_N20"]:
    run_colmap(name)

In [ ]:
import json as _json

for name in ["raw_dense", "raw_N30", "raw_N20"]:
    tf = f"{D_ABO}/{name}_processed/transforms.json"
    if os.path.exists(tf):
        n = len(_json.load(open(tf))["frames"])
        print(name, "-> зарегистрировано кадров:", n)
    else:
        print(name, "-> transforms.json нет (COLMAP не смог зарегистрировать ни одного кадра)")

In [ ]:
# Предполагается, что ячейки из Teil 4 (gdino_boxes, sam_union, модели GD/SAM2) уже выполнены
OBJECT_PROMPT_ABO = "a chair."

def make_masks(processed_dir, prompt=OBJECT_PROMPT_ABO):
    img_dir = f"{processed_dir}/images"
    mask_dir = f"{processed_dir}/masks"
    os.makedirs(mask_dir, exist_ok=True)
    names = sorted(os.listdir(img_dir))
    no_box = []
    for n in names:
        img = Image.open(f"{img_dir}/{n}").convert("RGB")
        W, H = img.size
        b, s = gdino_boxes(img, prompt, 0.3)
        if len(b) == 0:
            b, s = gdino_boxes(img, "furniture. object.", 0.22)
        keep = [bb for bb in b if 0.02 < ((bb[2]-bb[0])*(bb[3]-bb[1]))/(W*H) < 0.99]
        if not keep:
            no_box.append(n)
            mask = np.ones((H, W), bool)
        else:
            mask = sam_union(img, keep)
        Image.fromarray((mask*255).astype(np.uint8)).save(f"{mask_dir}/{os.path.splitext(n)[0]}.png")
    print(processed_dir, "-> масок:", len(os.listdir(mask_dir)), "| без бокса:", len(no_box), no_box[:5])

for name in ["raw_dense", "raw_N30", "raw_N20"]:
    pd = f"{D_ABO}/{name}_processed"
    if os.path.exists(f"{pd}/transforms.json"):
        make_masks(pd)

In [ ]:
def train_splatfacto(processed_dir, out_name, iters):
    cmd = (
        f"yes | TORCHDYNAMO_DISABLE=1 /content/nsenv/bin/ns-train splatfacto "
        f"--data {processed_dir} --max-num-iterations {iters} --steps-per-save 1000 "
        f"--vis tensorboard --viewer.quit-on-train-completion True "
        f"--output-dir {D_ABO}/outputs_{out_name} --pipeline.model.cull-alpha-thresh 0.1 "
        f"nerfstudio-data --masks-path masks"
    )
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(out_name, "-> returncode", r.returncode)
    print(r.stdout[-1500:])
    if r.returncode:
        print(r.stderr[-1500:])

train_splatfacto(f"{D_ABO}/raw_dense_processed", "dense", iters=7000)
train_splatfacto(f"{D_ABO}/raw_N30_processed", "n30", iters=3000)
train_splatfacto(f"{D_ABO}/raw_N20_processed", "n20", iters=3000)

In [ ]:
import glob

def export_and_mesh(out_name):
    cfgs = glob.glob(f"{D_ABO}/outputs_{out_name}/*/splatfacto/*/config.yml")
    if not cfgs:
        print(out_name, "-> config.yml не найден, тренировка не удалась")
        return
    cfg = sorted(cfgs)[-1]
    export_dir = f"{D_ABO}/exports_{out_name}"
    subprocess.run(f"/content/nsenv/bin/ns-export gaussian-splat --load-config {cfg} --output-dir {export_dir}", shell=True)

    script = f'''
import numpy as np
import open3d as o3d
import trimesh

pcd = o3d.io.read_point_cloud("{export_dir}/splat.ply")
pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
nn = o3d.geometry.KDTreeFlann(pcd)
step = max(1, len(pcd.points) // 2000)
d = []
for i in range(0, len(pcd.points), step):
    _, idx, dist2 = nn.search_knn_vector_3d(pcd.points[i], 2)
    d.append(np.sqrt(dist2[1]))
eps = 5 * float(np.median(d))
labels = np.array(pcd.cluster_dbscan(eps=eps, min_points=10))
if labels.max() >= 0:
    biggest = np.bincount(labels[labels >= 0]).argmax()
    pcd = pcd.select_by_index(np.where(labels == biggest)[0])
pcd.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
pcd.orient_normals_consistent_tangent_plane(30)
mesh, dens = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=9)
dens = np.asarray(dens)
mesh.remove_vertices_by_mask(dens <= np.quantile(dens, 0.1))
mesh.remove_degenerate_triangles(); mesh.remove_duplicated_vertices()
mesh.remove_duplicated_triangles(); mesh.remove_non_manifold_edges()
mesh.compute_triangle_normals()
out = "{export_dir}/chair_object.stl"
o3d.io.write_triangle_mesh(out, mesh)
m = trimesh.load(out)
m.fill_holes(); m.fix_normals(); m.process(validate=True)
m.export(out)
print("STL:", out, "| Ecken:", len(m.vertices), "| Dreiecke:", len(m.faces))
print("watertight:", m.is_watertight)
'''
    open(f"/content/mesh_abo_{out_name}.py", "w").write(script)
    r = subprocess.run(["/content/nsenv/bin/python", f"/content/mesh_abo_{out_name}.py"], capture_output=True, text=True)
    print(r.stdout)
    if r.returncode:
        print(r.stderr[-2000:])

for name in ["dense", "n30", "n20"]:
    export_and_mesh(name)

### 9b. Chamfer-дистанция против настоящей CAD-модели

Сравниваю каждую реконструкцию (dense/n30/n20) со скачанной ABO CAD-моделью
(`B07F2X8K62.glb`) — настоящий внешний ground truth вместо proxy из PROJECT_STATUS.md,
раздел 5. Метод тот же: нормализация масштаба, перебор поворотов 0-345° с шагом 15°,
ICP-доводка.

In [ ]:
r = subprocess.run(["/content/nsenv/bin/pip", "install", "-q", "pygltflib"], capture_output=True, text=True)
print(r.returncode, r.stderr[-500:] if r.returncode else "pygltflib ok")

In [ ]:
script = '''
import numpy as np
import open3d as o3d
import trimesh

def load_pts(path, n=30000):
    m = trimesh.load(path, force="mesh")
    pts, _ = trimesh.sample.sample_surface(m, n)
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pts)
    return pcd

def normalize(pcd):
    pts = np.asarray(pcd.points)
    c = pts.mean(0)
    pts = pts - c
    scale = np.linalg.norm(pts, axis=1).max()
    pts = pts / scale
    out = o3d.geometry.PointCloud()
    out.points = o3d.utility.Vector3dVector(pts)
    return out

def chamfer_vs_ref(ref, cand_path, label):
    cand = normalize(load_pts(cand_path))
    best = None
    for deg in range(0, 360, 15):
        R = o3d.geometry.get_rotation_matrix_from_axis_angle([0, 0, np.deg2rad(deg)])
        c2 = o3d.geometry.PointCloud(cand)
        c2.rotate(R, center=(0, 0, 0))
        reg = o3d.pipelines.registration.registration_icp(
            c2, ref, 0.05, np.eye(4),
            o3d.pipelines.registration.TransformationEstimationPointToPoint())
        if best is None or reg.inlier_rmse < best[0]:
            best = (reg.inlier_rmse, deg, reg.transformation, R)
    rmse, deg, T, R0 = best
    aligned = o3d.geometry.PointCloud(cand)
    aligned.rotate(R0, center=(0, 0, 0))
    aligned.transform(T)
    reg2 = o3d.pipelines.registration.registration_icp(
        aligned, ref, 0.03, np.eye(4),
        o3d.pipelines.registration.TransformationEstimationPointToPoint())
    aligned.transform(reg2.transformation)
    d1 = np.asarray(aligned.compute_point_cloud_distance(ref))
    d2 = np.asarray(ref.compute_point_cloud_distance(aligned))
    chamfer = d1.mean() + d2.mean()
    print(label, ": best angle", deg, "deg, ICP inlier RMSE %.4f" % rmse, ", Chamfer %.4f" % chamfer)
    return chamfer

ref_cad = normalize(load_pts("/content/abo_chair/cad/B07F2X8K62.glb"))
for name in ["dense", "n30", "n20"]:
    path = "/content/abo_chair/exports_" + name + "/chair_object.stl"
    import os as _os
    if _os.path.exists(path):
        chamfer_vs_ref(ref_cad, path, name + " vs. ABO-CAD")
    else:
        print(name, "-> STL нет, пропуск")
'''
open('/content/chamfer_vs_cad.py', 'w').write(script)
r = subprocess.run(['/content/nsenv/bin/python', '/content/chamfer_vs_cad.py'], capture_output=True, text=True)
print(r.stdout)
if r.returncode:
    print('STDERR:', r.stderr[-3000:])

In [ ]:
import shutil
os.makedirs('/content/drive/MyDrive/nerf_cache/abo_chair', exist_ok=True)
for name in ["dense", "n30", "n20"]:
    src_stl = f"{D_ABO}/exports_{name}/chair_object.stl"
    if os.path.exists(src_stl):
        shutil.copy(src_stl, f"/content/drive/MyDrive/nerf_cache/abo_chair/chair_object_{name}.stl")
print("сохранено в Drive:", os.listdir('/content/drive/MyDrive/nerf_cache/abo_chair'))

## 10. Повтор: n=20 sparse-view тест на truck (COLMAP нестабилен, пробуем ещё раз)

PROJECT_STATUS.md, раздел 8, пункт 3: в прошлый раз n=20 зарегистрировал только 4/20 кадров
— слишком мало для реконструкции. Набор фото тот же (`random.seed(0)` в ячейке 20/21), но
COLMAP недетерминирован — сопоставление признаков и RANSAC каждый раз идут по-разному, так
что просто пробуем ещё раз. Ниже до 5 автоматических попыток подряд, берём лучшую и считаем
на ней ту же Chamfer-дистанцию, что и для n=30.

Нужно, чтобы `/content/sparse_test/raw` уже содержал 20 фото (ячейка 20/21) и чтобы модели/
функции Teil 4 (`gdino_boxes`, `sam_union`, `OBJECT_PROMPT`) были уже выполнены.

In [ ]:
import json as _json, subprocess, os, shutil

RAW20 = '/content/sparse_test/raw'
PROCESSED20 = '/content/sparse_test/processed'
BEST20 = '/content/sparse_test/processed_best'

def try_colmap_once():
    subprocess.run(f'rm -rf {PROCESSED20}', shell=True)
    cmd = (
        f"QT_QPA_PLATFORM=offscreen /content/nsenv/bin/ns-process-data images "
        f"--data {RAW20} --output-dir {PROCESSED20} --matching-method exhaustive --no-gpu"
    )
    subprocess.run(cmd, shell=True, capture_output=True, text=True)
    tf = f'{PROCESSED20}/transforms.json'
    if os.path.exists(tf):
        return len(_json.load(open(tf))['frames'])
    return 0

best_n = -1
MAX_TRIES = 5
for attempt in range(1, MAX_TRIES + 1):
    n_reg = try_colmap_once()
    print(f"попытка {attempt}: зарегистрировано {n_reg}/20")
    if n_reg > best_n:
        best_n = n_reg
        if os.path.isdir(BEST20):
            shutil.rmtree(BEST20)
        shutil.copytree(PROCESSED20, BEST20)
    if n_reg >= 15:  # сопоставимо с n=30 (22/30 ~ 73%), дальше можно не пытаться
        print("достаточно кадров, останавливаемся")
        break

print(f"итог: лучшая попытка {best_n}/20 -> {BEST20}")
if best_n < 8:
    print("ВНИМАНИЕ: даже лучшая попытка слишком слабая для осмысленной реконструкции, "
          "смотри секцию 4b PROJECT_STATUS.md — увеличить MAX_TRIES или взять другой seed.")

In [ ]:
# маски: переиспользуем gdino_boxes/sam_union/OBJECT_PROMPT из Teil 4 ("a truck.")
img_dir = f'{BEST20}/images'
mask_dir = f'{BEST20}/masks'
os.makedirs(mask_dir, exist_ok=True)
names = sorted(os.listdir(img_dir))
no_box = []
for n in names:
    img = Image.open(f"{img_dir}/{n}").convert("RGB")
    W, H = img.size
    b, s = gdino_boxes(img, OBJECT_PROMPT, 0.3)
    if len(b) == 0:
        b, s = gdino_boxes(img, "object. vehicle.", 0.22)
    keep = [bb for bb in b if 0.02 < ((bb[2]-bb[0])*(bb[3]-bb[1]))/(W*H) < 0.99]
    if not keep:
        no_box.append(n)
        mask = np.ones((H, W), bool)
    else:
        mask = sam_union(img, keep)
    Image.fromarray((mask*255).astype(np.uint8)).save(f"{mask_dir}/{os.path.splitext(n)[0]}.png")
print("масок:", len(os.listdir(mask_dir)), "| без бокса:", len(no_box), no_box[:5])

In [ ]:
cmd = (
    f"yes | TORCHDYNAMO_DISABLE=1 /content/nsenv/bin/ns-train splatfacto "
    f"--data {BEST20} --max-num-iterations 3000 --steps-per-save 1000 "
    f"--vis tensorboard --viewer.quit-on-train-completion True "
    f"--output-dir outputs_sparse20 --pipeline.model.cull-alpha-thresh 0.1 "
    f"nerfstudio-data --masks-path masks"
)
r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
print("returncode:", r.returncode)
print(r.stdout[-1500:])
if r.returncode:
    print(r.stderr[-1500:])

In [ ]:
import glob

cfgs = glob.glob("outputs_sparse20/*/splatfacto/*/config.yml")
if not cfgs:
    print("config.yml не найден, тренировка не удалась")
else:
    cfg = sorted(cfgs)[-1]
    subprocess.run(f"/content/nsenv/bin/ns-export gaussian-splat --load-config {cfg} --output-dir exports/sparse20", shell=True)

    script = '''
import numpy as np
import open3d as o3d
import trimesh

pcd = o3d.io.read_point_cloud("exports/sparse20/splat.ply")
pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
nn = o3d.geometry.KDTreeFlann(pcd)
step = max(1, len(pcd.points) // 2000)
d = []
for i in range(0, len(pcd.points), step):
    _, idx, dist2 = nn.search_knn_vector_3d(pcd.points[i], 2)
    d.append(np.sqrt(dist2[1]))
eps = 5 * float(np.median(d))
labels = np.array(pcd.cluster_dbscan(eps=eps, min_points=10))
if labels.max() >= 0:
    biggest = np.bincount(labels[labels >= 0]).argmax()
    pcd = pcd.select_by_index(np.where(labels == biggest)[0])
pcd.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
pcd.orient_normals_consistent_tangent_plane(30)
mesh, dens = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=9)
dens = np.asarray(dens)
mesh.remove_vertices_by_mask(dens <= np.quantile(dens, 0.1))
mesh.remove_degenerate_triangles(); mesh.remove_duplicated_vertices()
mesh.remove_duplicated_triangles(); mesh.remove_non_manifold_edges()
mesh.compute_triangle_normals()
out = "exports/sparse20/sparse20_object.stl"
o3d.io.write_triangle_mesh(out, mesh)
m = trimesh.load(out)
m.fill_holes(); m.fix_normals(); m.process(validate=True)
m.export(out)
print("STL:", out, "| Ecken:", len(m.vertices), "| Dreiecke:", len(m.faces))
print("watertight:", m.is_watertight)
'''
    open('/content/mesh_sparse20.py', 'w').write(script)
    r = subprocess.run(['/content/nsenv/bin/python', '/content/mesh_sparse20.py'], capture_output=True, text=True)
    print(r.stdout)
    if r.returncode:
        print(r.stderr[-2000:])

In [ ]:
script = '''
import numpy as np
import open3d as o3d
import trimesh

def load_pts(path, n=30000):
    m = trimesh.load(path)
    pts, _ = trimesh.sample.sample_surface(m, n)
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pts)
    return pcd

def normalize(pcd):
    pts = np.asarray(pcd.points)
    c = pts.mean(0)
    pts = pts - c
    scale = np.linalg.norm(pts, axis=1).max()
    pts = pts / scale
    out = o3d.geometry.PointCloud()
    out.points = o3d.utility.Vector3dVector(pts)
    return out

ref = normalize(load_pts("exports/truck_masked/truck_object_ref.stl"))
sparse = normalize(load_pts("exports/sparse20/sparse20_object.stl"))

best = None
for deg in range(0, 360, 15):
    R = o3d.geometry.get_rotation_matrix_from_axis_angle([0, 0, np.deg2rad(deg)])
    cand = o3d.geometry.PointCloud(sparse)
    cand.rotate(R, center=(0, 0, 0))
    reg = o3d.pipelines.registration.registration_icp(
        cand, ref, 0.05, np.eye(4),
        o3d.pipelines.registration.TransformationEstimationPointToPoint())
    if best is None or reg.inlier_rmse < best[0]:
        best = (reg.inlier_rmse, deg, reg.transformation, R)

rmse, deg, T, R0 = best
print("лучший начальный угол:", deg, "| rmse после ICP:", rmse)

sparse_aligned = o3d.geometry.PointCloud(sparse)
sparse_aligned.rotate(R0, center=(0, 0, 0))
sparse_aligned.transform(T)

reg2 = o3d.pipelines.registration.registration_icp(
    sparse_aligned, ref, 0.03, np.eye(4),
    o3d.pipelines.registration.TransformationEstimationPointToPoint())
sparse_aligned.transform(reg2.transformation)

d1 = np.asarray(sparse_aligned.compute_point_cloud_distance(ref))
d2 = np.asarray(ref.compute_point_cloud_distance(sparse_aligned))
chamfer = d1.mean() + d2.mean()
print(f"Chamfer distance n=20 (нормализовано на размер объекта): {chamfer:.4f}")
print(f"  -> sparse20 к ref: {d1.mean():.4f} (медиана {np.median(d1):.4f})")
print(f"  -> ref к sparse20: {d2.mean():.4f} (медиана {np.median(d2):.4f})")
'''
open('/content/chamfer20.py', 'w').write(script)
r = subprocess.run(['/content/nsenv/bin/python', '/content/chamfer20.py'], capture_output=True, text=True)
print(r.stdout)
if r.returncode:
    print('STDERR:', r.stderr[-3000:])

In [ ]:
import os, subprocess, sys

ABO_ITEM = "B07F2X8K62"
ABO_SPIN_ID = "f8022078"
ABO_SPIN_PREFIX = ABO_SPIN_ID[:2]
ABO_N_FRAMES = 72
D_ABO = "/content/abo_chair"
os.makedirs(f"{D_ABO}/raw", exist_ok=True)

base = "https://amazon-berkeley-objects.s3.amazonaws.com"
for az in range(ABO_N_FRAMES):
    fn = f"{ABO_SPIN_ID}_{az:02d}.jpg"
    url = f"{base}/spins/original/{ABO_SPIN_PREFIX}/{ABO_SPIN_ID}/{fn}"
    out = f"{D_ABO}/raw/{fn}"
    if not os.path.exists(out):
        r = subprocess.run(["curl", "-s", "-L", "-w", "%{http_code}", url, "-o", out], capture_output=True, text=True)
        print(az, fn, "http", r.stdout.strip())
        sys.stdout.flush()
print("фото скачано:", len(os.listdir(f"{D_ABO}/raw")))

cad_dir = f"{D_ABO}/cad"
os.makedirs(cad_dir, exist_ok=True)
cad_url = f"{base}/3dmodels/original/{ABO_ITEM[-1]}/{ABO_ITEM}.glb"
r = subprocess.run(["curl", "-s", "-L", "-w", "%{http_code}", cad_url, "-o", f"{cad_dir}/{ABO_ITEM}.glb"], capture_output=True, text=True)
print("CAD http", r.stdout.strip(), "->", os.listdir(cad_dir))


In [ ]:
import shutil
import numpy as np

def make_subset(n, raw_dir=f"{D_ABO}/raw", out_name=None):
    out_name = out_name or f"raw_N{n}"
    out_dir = f"{D_ABO}/{out_name}"
    if os.path.isdir(out_dir):
        shutil.rmtree(out_dir)
    os.makedirs(out_dir)
    names = sorted(os.listdir(raw_dir))
    idx = sorted(set(np.linspace(0, len(names) - 1, n).round().astype(int).tolist()))
    for i in idx:
        shutil.copy(f"{raw_dir}/{names[i]}", out_dir)
    print(out_name, "->", len(os.listdir(out_dir)), "фото")

make_subset(ABO_N_FRAMES, out_name="raw_dense")  # полная turntable-серия как референс
make_subset(30)
make_subset(20)


In [ ]:
!apt-get -qq update
!apt-get -qq install -y colmap
!colmap -h | head -3


In [ ]:
def run_colmap(subset_dir_name):
    raw = f"{D_ABO}/{subset_dir_name}"
    processed = f"{D_ABO}/{subset_dir_name}_processed"
    subprocess.run(f"rm -rf {processed}", shell=True)
    cmd = (
        f"QT_QPA_PLATFORM=offscreen /content/nsenv/bin/ns-process-data images "
        f"--data {raw} --output-dir {processed} --matching-method exhaustive --no-gpu"
    )
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(subset_dir_name, "-> returncode", r.returncode)
    print(r.stdout[-1500:])
    if r.returncode:
        print(r.stderr[-1500:])

for name in ["raw_dense", "raw_N30", "raw_N20"]:
    run_colmap(name)


In [ ]:
!ls -la /content/nsenv/bin/ 2>&1 | head -20
!ls -la /content/ 2>&1


In [ ]:
%%bash
set -e
NSPY=/content/nsenv/bin/python

if [ -x /content/nsenv/bin/ns-train ]; then
  echo '=== venv уже собран, проверка ==='
  $NSPY -c "import numpy,torch,gsplat; from nerfstudio.models.splatfacto import SplatfactoModel; print('numpy',numpy.__version__,'| torch',torch.__version__,'| gsplat',gsplat.__version__,'| cuda',torch.cuda.is_available(),'| splatfacto OK')"
  exit 0
fi

pip install -q uv
uv python install 3.10
uv venv --python 3.10 /content/nsenv

uv pip install --python $NSPY "torch==2.1.2" "torchvision==0.16.2" --index-url https://download.pytorch.org/whl/cu118
uv pip install --python $NSPY "setuptools<70" wheel
uv pip install --python $NSPY "gsplat==1.4.0+pt21cu118" --index-url https://docs.gsplat.studio/whl/pt21cu118 --extra-index-url https://pypi.org/simple --index-strategy unsafe-best-match
uv pip install --python $NSPY "nerfstudio==1.1.5" open3d trimesh
uv pip install --python $NSPY "numpy==1.26.4"

echo '=== проверка ==='
$NSPY -c "import numpy,torch,gsplat; from gsplat.cuda._backend import _C; from nerfstudio.models.splatfacto import SplatfactoModel; print('numpy',numpy.__version__,'| torch',torch.__version__,'| gsplat',gsplat.__version__,'| cuda',torch.cuda.is_available(),'| splatfacto OK')"
/content/nsenv/bin/ns-train --help >/dev/null 2>&1 && echo 'ns-train: OK' || echo 'ns-train: FAIL'


In [ ]:
print(gd_model.device)


In [ ]:
!pip install -q -U transformers
import torch, numpy as np
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, Sam2Model, Sam2Processor

dev = "cuda"
OBJECT_PROMPT = "a truck."  # текстовое описание объекта пользователя — заменить под реальный сценарий
GD, SAM = "IDEA-Research/grounding-dino-base", "facebook/sam2-hiera-large"
gd_proc = AutoProcessor.from_pretrained(GD)
gd_model = AutoModelForZeroShotObjectDetection.from_pretrained(GD).to(dev).eval()
sam_proc = Sam2Processor.from_pretrained(SAM)
sam_model = Sam2Model.from_pretrained(SAM).to(dev).eval()
print("модели ок (SAM2)")

def gdino_boxes(img, text=OBJECT_PROMPT, thr=0.3):
    inp = gd_proc(images=img, text=text, return_tensors="pt").to(dev)
    with torch.no_grad():
        out = gd_model(**inp)
    tgt = [img.size[::-1]]
    try:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    except TypeError:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, box_threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    return r["boxes"].cpu().numpy(), r["scores"].cpu().numpy()


In [ ]:
!colmap -h | head -3


In [ ]:
!pip install -q -U transformers
import torch, numpy as np
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, Sam2Model, Sam2Processor

dev = "cuda"
OBJECT_PROMPT = "a truck."
GD, SAM = "IDEA-Research/grounding-dino-base", "facebook/sam2-hiera-large"
gd_proc = AutoProcessor.from_pretrained(GD)
gd_model = AutoModelForZeroShotObjectDetection.from_pretrained(GD).to(dev).eval()
sam_proc = Sam2Processor.from_pretrained(SAM)
sam_model = Sam2Model.from_pretrained(SAM).to(dev).eval()
print("модели ок (SAM2)")

def gdino_boxes(img, text=OBJECT_PROMPT, thr=0.3):
    inp = gd_proc(images=img, text=text, return_tensors="pt").to(dev)
    with torch.no_grad():
        out = gd_model(**inp)
    tgt = [img.size[::-1]]
    try:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    except TypeError:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, box_threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    return r["boxes"].cpu().numpy(), r["scores"].cpu().numpy()


In [ ]:
!colmap -h | head -3


In [ ]:
import os, subprocess

D_ABO = "/content/abo_chair"  # на случай если переменная тоже потерялась

def run_colmap(subset_dir_name):
    raw = f"{D_ABO}/{subset_dir_name}"
    processed = f"{D_ABO}/{subset_dir_name}_processed"
    subprocess.run(f"rm -rf {processed}", shell=True)
    cmd = (
        f"QT_QPA_PLATFORM=offscreen /content/nsenv/bin/ns-process-data images "
        f"--data {raw} --output-dir {processed} --matching-method exhaustive --no-gpu"
    )
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(subset_dir_name, "-> returncode", r.returncode)
    print(r.stdout[-1500:])
    if r.returncode:
        print(r.stderr[-1500:])

for name in ["raw_dense", "raw_N30", "raw_N20"]:
    run_colmap(name)


In [ ]:
def run_colmap_seq(subset_dir_name, matching="sequential"):
    raw = f"{D_ABO}/{subset_dir_name}"
    processed = f"{D_ABO}/{subset_dir_name}_processed"
    subprocess.run(f"rm -rf {processed}", shell=True)
    cmd = (
        f"QT_QPA_PLATFORM=offscreen /content/nsenv/bin/ns-process-data images "
        f"--data {raw} --output-dir {processed} --matching-method {matching} --no-gpu"
    )
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(subset_dir_name, matching, "-> returncode", r.returncode)
    print(r.stdout[-2000:])
    if r.returncode:
        print(r.stderr[-1500:])

run_colmap_seq("raw_dense", "sequential")


In [ ]:
import numpy as np, json, os, shutil
from PIL import Image

def pose_spherical(theta_deg, phi_deg, radius):
    def trans_t(t):
        return np.array([[1,0,0,0],[0,1,0,0],[0,0,1,t],[0,0,0,1]], dtype=float)
    def rot_phi(p):
        p = np.deg2rad(p)
        return np.array([[1,0,0,0],[0,np.cos(p),-np.sin(p),0],[0,np.sin(p),np.cos(p),0],[0,0,0,1]], dtype=float)
    def rot_theta(t):
        t = np.deg2rad(t)
        return np.array([[np.cos(t),0,-np.sin(t),0],[0,1,0,0],[np.sin(t),0,np.cos(t),0],[0,0,0,1]], dtype=float)
    c2w = trans_t(radius)
    c2w = rot_phi(phi_deg) @ c2w
    c2w = rot_theta(theta_deg) @ c2w
    c2w = np.array([[-1,0,0,0],[0,0,1,0],[0,1,0,0],[0,0,0,1]], dtype=float) @ c2w
    return c2w

def build_synthetic_dataset(subset_dir_name, radius=2.5, elevation_deg=-15.0, fov_deg=40.0):
    raw_dir = f"{D_ABO}/{subset_dir_name}"
    names = sorted(os.listdir(raw_dir))
    w, h = Image.open(f"{raw_dir}/{names[0]}").size

    out_dir = f"{D_ABO}/{subset_dir_name}_synth"
    img_out = f"{out_dir}/images"
    os.makedirs(img_out, exist_ok=True)

    fov_x = np.deg2rad(fov_deg)
    fl = 0.5 * w / np.tan(0.5 * fov_x)

    frames = []
    for n in names:
        az = int(n.split('_')[1].split('.')[0])
        theta = az * 5.0
        c2w = pose_spherical(theta, elevation_deg, radius)
        dst = f"{img_out}/{n}"
        if not os.path.exists(dst):
            shutil.copy(f"{raw_dir}/{n}", dst)
        frames.append({"file_path": f"images/{n}", "transform_matrix": c2w.tolist()})

    out = {
        "fl_x": fl, "fl_y": fl,
        "cx": w / 2, "cy": h / 2,
        "w": w, "h": h,
        "camera_model": "OPENCV",
        "k1": 0.0, "k2": 0.0, "p1": 0.0, "p2": 0.0,
        "frames": frames,
    }
    with open(f"{out_dir}/transforms.json", "w") as f:
        json.dump(out, f, indent=2)
    print(subset_dir_name, "-> синтетические позы:", len(frames), "->", out_dir)

for name in ["raw_dense", "raw_N30", "raw_N20"]:
    build_synthetic_dataset(name)


In [ ]:
OBJECT_PROMPT_ABO = "a chair."

def make_masks(processed_dir, prompt=OBJECT_PROMPT_ABO):
    img_dir = f"{processed_dir}/images"
    mask_dir = f"{processed_dir}/masks"
    os.makedirs(mask_dir, exist_ok=True)
    names = sorted(os.listdir(img_dir))
    no_box = []
    for n in names:
        img = Image.open(f"{img_dir}/{n}").convert("RGB")
        W, H = img.size
        b, s = gdino_boxes(img, prompt, 0.3)
        if len(b) == 0:
            b, s = gdino_boxes(img, "furniture. object.", 0.22)
        keep = [bb for bb in b if 0.02 < ((bb[2]-bb[0])*(bb[3]-bb[1]))/(W*H) < 0.99]
        if not keep:
            no_box.append(n)
            mask = np.ones((H, W), bool)
        else:
            mask = sam_union(img, keep)
        Image.fromarray((mask*255).astype(np.uint8)).save(f"{mask_dir}/{os.path.splitext(n)[0]}.png")
    print(processed_dir, "-> масок:", len(os.listdir(mask_dir)), "| без бокса:", len(no_box), no_box[:5])

for name in ["raw_dense", "raw_N30", "raw_N20"]:
    make_masks(f"{D_ABO}/{name}_synth")


In [ ]:
_pp = getattr(sam_proc, "post_process_masks", None) or sam_proc.image_processor.post_process_masks

def sam_union(img, boxes):
    inp = sam_proc(img, input_boxes=[[list(map(float, b)) for b in boxes]], return_tensors="pt").to(dev)
    with torch.no_grad():
        out = sam_model(**inp, multimask_output=True)
    masks = _pp(out.pred_masks.cpu(), inp["original_sizes"].cpu())[0]
    m = masks.numpy()
    iou = out.iou_scores.cpu().numpy()[0]
    sel = np.stack([m[i, iou[i].argmax()] for i in range(m.shape[0])])
    return sel.any(0)

print("sam_union определена")


In [ ]:
OBJECT_PROMPT_ABO = "a chair."

def make_masks(processed_dir, prompt=OBJECT_PROMPT_ABO):
    img_dir = f"{processed_dir}/images"
    mask_dir = f"{processed_dir}/masks"
    os.makedirs(mask_dir, exist_ok=True)
    names = sorted(os.listdir(img_dir))
    no_box = []
    for n in names:
        img = Image.open(f"{img_dir}/{n}").convert("RGB")
        W, H = img.size
        b, s = gdino_boxes(img, prompt, 0.3)
        if len(b) == 0:
            b, s = gdino_boxes(img, "furniture. object.", 0.22)
        keep = [bb for bb in b if 0.02 < ((bb[2]-bb[0])*(bb[3]-bb[1]))/(W*H) < 0.99]
        if not keep:
            no_box.append(n)
            mask = np.ones((H, W), bool)
        else:
            mask = sam_union(img, keep)
        Image.fromarray((mask*255).astype(np.uint8)).save(f"{mask_dir}/{os.path.splitext(n)[0]}.png")
    print(processed_dir, "-> масок:", len(os.listdir(mask_dir)), "| без бокса:", len(no_box), no_box[:5])

for name in ["raw_dense", "raw_N30", "raw_N20"]:
    make_masks(f"{D_ABO}/{name}_synth")


In [ ]:
def train_splatfacto(processed_dir, out_name, iters):
    cmd = (
        f"yes | TORCHDYNAMO_DISABLE=1 /content/nsenv/bin/ns-train splatfacto "
        f"--data {processed_dir} --max-num-iterations {iters} --steps-per-save 1000 "
        f"--vis tensorboard --viewer.quit-on-train-completion True "
        f"--output-dir {D_ABO}/outputs_{out_name} --pipeline.model.cull-alpha-thresh 0.1 "
        f"nerfstudio-data --masks-path masks"
    )
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(out_name, "-> returncode", r.returncode)
    print(r.stdout[-2000:])
    if r.returncode:
        print(r.stderr[-2000:])

train_splatfacto(f"{D_ABO}/raw_dense_synth", "dense", iters=7000)
train_splatfacto(f"{D_ABO}/raw_N30_synth", "n30", iters=3000)
train_splatfacto(f"{D_ABO}/raw_N20_synth", "n20", iters=3000)


In [ ]:
def add_mask_paths(out_name):
    d = f"{D_ABO}/{out_name}_synth"
    tf = f"{d}/transforms.json"
    data = json.load(open(tf))
    for fr in data["frames"]:
        fname = os.path.basename(fr["file_path"])
        stem = os.path.splitext(fname)[0]
        mask_file = f"masks/{stem}.png"
        if os.path.exists(f"{d}/{mask_file}"):
            fr["mask_path"] = mask_file
    json.dump(data, open(tf, "w"), indent=2)
    n_with_mask = sum(1 for fr in data["frames"] if "mask_path" in fr)
    print(out_name, "-> масок привязано:", n_with_mask, "/", len(data["frames"]))

for name in ["raw_dense", "raw_N30", "raw_N20"]:
    add_mask_paths(name)


In [ ]:
def train_splatfacto(processed_dir, out_name, iters):
    cmd = (
        f"yes | TORCHDYNAMO_DISABLE=1 /content/nsenv/bin/ns-train splatfacto "
        f"--data {processed_dir} --max-num-iterations {iters} --steps-per-save 1000 "
        f"--vis tensorboard --viewer.quit-on-train-completion True "
        f"--output-dir {D_ABO}/outputs_{out_name} --pipeline.model.cull-alpha-thresh 0.1 "
        f"nerfstudio-data"
    )
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(out_name, "-> returncode", r.returncode)
    print(r.stdout[-2000:])
    if r.returncode:
        print(r.stderr[-2000:])

train_splatfacto(f"{D_ABO}/raw_dense_synth", "dense", iters=7000)
train_splatfacto(f"{D_ABO}/raw_N30_synth", "n30", iters=3000)
train_splatfacto(f"{D_ABO}/raw_N20_synth", "n20", iters=3000)


In [1]:
import os
print("abo_chair:", os.path.exists('/content/abo_chair'))
print("venv:", os.path.exists('/content/nsenv/bin/ns-train'))
print("dense config:", os.path.exists('/content/abo_chair/outputs_dense'))
print("n30 config:", os.path.exists('/content/abo_chair/outputs_n30'))
print("n20 config:", os.path.exists('/content/abo_chair/outputs_n20'))


In [2]:
%%bash
set -e
NSPY=/content/nsenv/bin/python

if [ -x /content/nsenv/bin/ns-train ]; then
  echo '=== venv уже собран, проверка ==='
  $NSPY -c "import numpy,torch,gsplat; from nerfstudio.models.splatfacto import SplatfactoModel; print('numpy',numpy.__version__,'| torch',torch.__version__,'| gsplat',gsplat.__version__,'| cuda',torch.cuda.is_available(),'| splatfacto OK')"
  exit 0
fi

pip install -q uv
uv python install 3.10
uv venv --python 3.10 /content/nsenv

uv pip install --python $NSPY "torch==2.1.2" "torchvision==0.16.2" --index-url https://download.pytorch.org/whl/cu118
uv pip install --python $NSPY "setuptools<70" wheel
uv pip install --python $NSPY "gsplat==1.4.0+pt21cu118" --index-url https://docs.gsplat.studio/whl/pt21cu118 --extra-index-url https://pypi.org/simple --index-strategy unsafe-best-match
uv pip install --python $NSPY "nerfstudio==1.1.5" open3d trimesh
uv pip install --python $NSPY "numpy==1.26.4"

echo '=== проверка ==='
$NSPY -c "import numpy,torch,gsplat; from gsplat.cuda._backend import _C; from nerfstudio.models.splatfacto import SplatfactoModel; print('numpy',numpy.__version__,'| torch',torch.__version__,'| gsplat',gsplat.__version__,'| cuda',torch.cuda.is_available(),'| splatfacto OK')"
/content/nsenv/bin/ns-train --help >/dev/null 2>&1 && echo 'ns-train: OK' || echo 'ns-train: FAIL'


In [3]:
!pip install -q -U transformers
import torch, numpy as np
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, Sam2Model, Sam2Processor

dev = "cuda"
OBJECT_PROMPT = "a truck."
GD, SAM = "IDEA-Research/grounding-dino-base", "facebook/sam2-hiera-large"
gd_proc = AutoProcessor.from_pretrained(GD)
gd_model = AutoModelForZeroShotObjectDetection.from_pretrained(GD).to(dev).eval()
sam_proc = Sam2Processor.from_pretrained(SAM)
sam_model = Sam2Model.from_pretrained(SAM).to(dev).eval()
print("модели ок (SAM2)")

def gdino_boxes(img, text=OBJECT_PROMPT, thr=0.3):
    inp = gd_proc(images=img, text=text, return_tensors="pt").to(dev)
    with torch.no_grad():
        out = gd_model(**inp)
    tgt = [img.size[::-1]]
    try:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    except TypeError:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, box_threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    return r["boxes"].cpu().numpy(), r["scores"].cpu().numpy()

_pp = getattr(sam_proc, "post_process_masks", None) or sam_proc.image_processor.post_process_masks

def sam_union(img, boxes):
    inp = sam_proc(img, input_boxes=[[list(map(float, b)) for b in boxes]], return_tensors="pt").to(dev)
    with torch.no_grad():
        out = sam_model(**inp, multimask_output=True)
    masks = _pp(out.pred_masks.cpu(), inp["original_sizes"].cpu())[0]
    m = masks.numpy()
    iou = out.iou_scores.cpu().numpy()[0]
    sel = np.stack([m[i, iou[i].argmax()] for i in range(m.shape[0])])
    return sel.any(0)


In [4]:
import os, subprocess

ABO_ITEM = "B07F2X8K62"
ABO_SPIN_ID = "f8022078"
ABO_SPIN_PREFIX = ABO_SPIN_ID[:2]
ABO_N_FRAMES = 72
D_ABO = "/content/abo_chair"
os.makedirs(f"{D_ABO}/raw", exist_ok=True)

base = "https://amazon-berkeley-objects.s3.amazonaws.com"
for az in range(ABO_N_FRAMES):
    fn = f"{ABO_SPIN_ID}_{az:02d}.jpg"
    url = f"{base}/spins/original/{ABO_SPIN_PREFIX}/{ABO_SPIN_ID}/{fn}"
    out = f"{D_ABO}/raw/{fn}"
    if not os.path.exists(out):
        subprocess.run(["curl", "-s", "-L", url, "-o", out], check=True)
print("фото скачано:", len(os.listdir(f"{D_ABO}/raw")))

cad_dir = f"{D_ABO}/cad"
os.makedirs(cad_dir, exist_ok=True)
cad_url = f"{base}/3dmodels/original/{ABO_ITEM[-1]}/{ABO_ITEM}.glb"
subprocess.run(["curl", "-s", "-L", cad_url, "-o", f"{cad_dir}/{ABO_ITEM}.glb"], check=True)
print("CAD скачан:", os.listdir(cad_dir))


In [5]:
import shutil
import numpy as np

def make_subset(n, raw_dir=f"{D_ABO}/raw", out_name=None):
    out_name = out_name or f"raw_N{n}"
    out_dir = f"{D_ABO}/{out_name}"
    if os.path.isdir(out_dir):
        shutil.rmtree(out_dir)
    os.makedirs(out_dir)
    names = sorted(os.listdir(raw_dir))
    idx = sorted(set(np.linspace(0, len(names) - 1, n).round().astype(int).tolist()))
    for i in idx:
        shutil.copy(f"{raw_dir}/{names[i]}", out_dir)
    print(out_name, "->", len(os.listdir(out_dir)), "фото")

make_subset(ABO_N_FRAMES, out_name="raw_dense")
make_subset(30)
make_subset(20)


In [6]:
import json

def pose_spherical(theta_deg, phi_deg, radius):
    def trans_t(t):
        return np.array([[1,0,0,0],[0,1,0,0],[0,0,1,t],[0,0,0,1]], dtype=float)
    def rot_phi(p):
        p = np.deg2rad(p)
        return np.array([[1,0,0,0],[0,np.cos(p),-np.sin(p),0],[0,np.sin(p),np.cos(p),0],[0,0,0,1]], dtype=float)
    def rot_theta(t):
        t = np.deg2rad(t)
        return np.array([[np.cos(t),0,-np.sin(t),0],[0,1,0,0],[np.sin(t),0,np.cos(t),0],[0,0,0,1]], dtype=float)
    c2w = trans_t(radius)
    c2w = rot_phi(phi_deg) @ c2w
    c2w = rot_theta(theta_deg) @ c2w
    c2w = np.array([[-1,0,0,0],[0,0,1,0],[0,1,0,0],[0,0,0,1]], dtype=float) @ c2w
    return c2w

def build_synthetic_dataset(subset_dir_name, radius=2.5, elevation_deg=-15.0, fov_deg=40.0):
    raw_dir = f"{D_ABO}/{subset_dir_name}"
    names = sorted(os.listdir(raw_dir))
    w, h = Image.open(f"{raw_dir}/{names[0]}").size
    out_dir = f"{D_ABO}/{subset_dir_name}_synth"
    img_out = f"{out_dir}/images"
    os.makedirs(img_out, exist_ok=True)
    fov_x = np.deg2rad(fov_deg)
    fl = 0.5 * w / np.tan(0.5 * fov_x)
    frames = []
    for n in names:
        az = int(n.split('_')[1].split('.')[0])
        theta = az * 5.0
        c2w = pose_spherical(theta, elevation_deg, radius)
        dst = f"{img_out}/{n}"
        if not os.path.exists(dst):
            shutil.copy(f"{raw_dir}/{n}", dst)
        frames.append({"file_path": f"images/{n}", "transform_matrix": c2w.tolist()})
    out = {
        "fl_x": fl, "fl_y": fl, "cx": w / 2, "cy": h / 2, "w": w, "h": h,
        "camera_model": "OPENCV", "k1": 0.0, "k2": 0.0, "p1": 0.0, "p2": 0.0,
        "frames": frames,
    }
    with open(f"{out_dir}/transforms.json", "w") as f:
        json.dump(out, f, indent=2)
    print(subset_dir_name, "-> поз:", len(frames), "->", out_dir)

for name in ["raw_dense", "raw_N30", "raw_N20"]:
    build_synthetic_dataset(name)


In [7]:
OBJECT_PROMPT_ABO = "a chair."

def make_masks(processed_dir, prompt=OBJECT_PROMPT_ABO):
    img_dir = f"{processed_dir}/images"
    mask_dir = f"{processed_dir}/masks"
    os.makedirs(mask_dir, exist_ok=True)
    names = sorted(os.listdir(img_dir))
    no_box = []
    for n in names:
        img = Image.open(f"{img_dir}/{n}").convert("RGB")
        W, H = img.size
        b, s = gdino_boxes(img, prompt, 0.3)
        if len(b) == 0:
            b, s = gdino_boxes(img, "furniture. object.", 0.22)
        keep = [bb for bb in b if 0.02 < ((bb[2]-bb[0])*(bb[3]-bb[1]))/(W*H) < 0.99]
        if not keep:
            no_box.append(n)
            mask = np.ones((H, W), bool)
        else:
            mask = sam_union(img, keep)
        Image.fromarray((mask*255).astype(np.uint8)).save(f"{mask_dir}/{os.path.splitext(n)[0]}.png")
    print(processed_dir, "-> масок:", len(os.listdir(mask_dir)), "| без бокса:", len(no_box), no_box[:5])

for name in ["raw_dense", "raw_N30", "raw_N20"]:
    make_masks(f"{D_ABO}/{name}_synth")


In [8]:
def add_mask_paths(out_name):
    d = f"{D_ABO}/{out_name}_synth"
    tf = f"{d}/transforms.json"
    data = json.load(open(tf))
    for fr in data["frames"]:
        fname = os.path.basename(fr["file_path"])
        stem = os.path.splitext(fname)[0]
        mask_file = f"masks/{stem}.png"
        if os.path.exists(f"{d}/{mask_file}"):
            fr["mask_path"] = mask_file
    json.dump(data, open(tf, "w"), indent=2)
    n_with_mask = sum(1 for fr in data["frames"] if "mask_path" in fr)
    print(out_name, "-> масок привязано:", n_with_mask, "/", len(data["frames"]))

for name in ["raw_dense", "raw_N30", "raw_N20"]:
    add_mask_paths(name)


In [9]:
import subprocess

def train_splatfacto(processed_dir, out_name, iters):
    cmd = (
        f"yes | TORCHDYNAMO_DISABLE=1 /content/nsenv/bin/ns-train splatfacto "
        f"--data {processed_dir} --max-num-iterations {iters} --steps-per-save 1000 "
        f"--vis tensorboard --viewer.quit-on-train-completion True "
        f"--output-dir {D_ABO}/outputs_{out_name} --pipeline.model.cull-alpha-thresh 0.1 "
        f"nerfstudio-data"
    )
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(out_name, "-> returncode", r.returncode)
    print(r.stdout[-1500:])
    if r.returncode:
        print(r.stderr[-1500:])

train_splatfacto(f"{D_ABO}/raw_dense_synth", "dense", iters=7000)
train_splatfacto(f"{D_ABO}/raw_N30_synth", "n30", iters=3000)
train_splatfacto(f"{D_ABO}/raw_N20_synth", "n20", iters=3000)


In [10]:
import glob

def export_and_mesh(out_name):
    cfgs = glob.glob(f"{D_ABO}/outputs_{out_name}/*/splatfacto/*/config.yml")
    if not cfgs:
        print(out_name, "-> config.yml не найден")
        return
    cfg = sorted(cfgs)[-1]
    export_dir = f"{D_ABO}/exports_{out_name}"
    subprocess.run(f"/content/nsenv/bin/ns-export gaussian-splat --load-config {cfg} --output-dir {export_dir}", shell=True)

    script = f'''
import numpy as np
import open3d as o3d
import trimesh

pcd = o3d.io.read_point_cloud("{export_dir}/splat.ply")
pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
nn = o3d.geometry.KDTreeFlann(pcd)
step = max(1, len(pcd.points) // 2000)
d = []
for i in range(0, len(pcd.points), step):
    _, idx, dist2 = nn.search_knn_vector_3d(pcd.points[i], 2)
    d.append(np.sqrt(dist2[1]))
eps = 5 * float(np.median(d))
labels = np.array(pcd.cluster_dbscan(eps=eps, min_points=10))
if labels.max() >= 0:
    biggest = np.bincount(labels[labels >= 0]).argmax()
    pcd = pcd.select_by_index(np.where(labels == biggest)[0])
pcd.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
pcd.orient_normals_consistent_tangent_plane(30)
mesh, dens = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=9)
dens = np.asarray(dens)
mesh.remove_vertices_by_mask(dens <= np.quantile(dens, 0.1))
mesh.remove_degenerate_triangles(); mesh.remove_duplicated_vertices()
mesh.remove_duplicated_triangles(); mesh.remove_non_manifold_edges()
mesh.compute_triangle_normals()
out = "{export_dir}/chair_object.stl"
o3d.io.write_triangle_mesh(out, mesh)
m = trimesh.load(out)
m.fill_holes(); m.fix_normals(); m.process(validate=True)
m.export(out)
print("STL:", out, "| Ecken:", len(m.vertices), "| Dreiecke:", len(m.faces))
print("watertight:", m.is_watertight)
'''
    open(f"/content/mesh_abo_{out_name}.py", "w").write(script)
    r = subprocess.run(["/content/nsenv/bin/python", f"/content/mesh_abo_{out_name}.py"], capture_output=True, text=True)
    print(r.stdout)
    if r.returncode:
        print(r.stderr[-2000:])

for name in ["dense", "n30", "n20"]:
    export_and_mesh(name)


In [11]:
r = subprocess.run(["/content/nsenv/bin/pip", "install", "-q", "pygltflib"], capture_output=True, text=True)
print(r.returncode, r.stderr[-500:] if r.returncode else "pygltflib ok")


In [12]:
r = subprocess.run(["uv", "pip", "install", "-q", "--python", "/content/nsenv/bin/python", "pygltflib"], capture_output=True, text=True)
print(r.returncode)
print(r.stdout[-500:])
if r.returncode:
    print(r.stderr[-1000:])


In [13]:
script = '''
import numpy as np
import open3d as o3d
import trimesh

def load_pts(path, n=30000):
    m = trimesh.load(path, force="mesh")
    pts, _ = trimesh.sample.sample_surface(m, n)
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pts)
    return pcd

def normalize(pcd):
    pts = np.asarray(pcd.points)
    c = pts.mean(0)
    pts = pts - c
    scale = np.linalg.norm(pts, axis=1).max()
    pts = pts / scale
    out = o3d.geometry.PointCloud()
    out.points = o3d.utility.Vector3dVector(pts)
    return out

def chamfer_vs_ref(ref, cand_path, label):
    cand = normalize(load_pts(cand_path))
    best = None
    for deg in range(0, 360, 15):
        R = o3d.geometry.get_rotation_matrix_from_axis_angle([0, 0, np.deg2rad(deg)])
        c2 = o3d.geometry.PointCloud(cand)
        c2.rotate(R, center=(0, 0, 0))
        reg = o3d.pipelines.registration.registration_icp(
            c2, ref, 0.05, np.eye(4),
            o3d.pipelines.registration.TransformationEstimationPointToPoint())
        if best is None or reg.inlier_rmse < best[0]:
            best = (reg.inlier_rmse, deg, reg.transformation, R)
    rmse, deg, T, R0 = best
    aligned = o3d.geometry.PointCloud(cand)
    aligned.rotate(R0, center=(0, 0, 0))
    aligned.transform(T)
    reg2 = o3d.pipelines.registration.registration_icp(
        aligned, ref, 0.03, np.eye(4),
        o3d.pipelines.registration.TransformationEstimationPointToPoint())
    aligned.transform(reg2.transformation)
    d1 = np.asarray(aligned.compute_point_cloud_distance(ref))
    d2 = np.asarray(ref.compute_point_cloud_distance(aligned))
    chamfer = d1.mean() + d2.mean()
    print(label, ": best angle", deg, "deg, ICP inlier RMSE %.4f" % rmse, ", Chamfer %.4f" % chamfer)
    return chamfer

ref_cad = normalize(load_pts("/content/abo_chair/cad/B07F2X8K62.glb"))
for name in ["dense", "n30", "n20"]:
    path = "/content/abo_chair/exports_" + name + "/chair_object.stl"
    import os as _os
    if _os.path.exists(path):
        chamfer_vs_ref(ref_cad, path, name + " vs. ABO-CAD")
'''
open('/content/chamfer_vs_cad.py', 'w').write(script)
r = subprocess.run(['/content/nsenv/bin/python', '/content/chamfer_vs_cad.py'], capture_output=True, text=True)
print(r.stdout)
if r.returncode:
    print('STDERR:', r.stderr[-3000:])


In [14]:
from google.colab import drive
drive.mount('/content/drive')


In [1]:
from google.colab import files
files.download('/content/abo_chair/exports_dense/chair_object.stl')


In [2]:
import os
print("abo_chair:", os.path.exists('/content/abo_chair'))
print("exports_dense dir:", os.listdir('/content/abo_chair') if os.path.exists('/content/abo_chair') else "нет abo_chair")


In [ ]:
%%bash
set -e
NSPY=/content/nsenv/bin/python

if [ -x /content/nsenv/bin/ns-train ]; then
  echo '=== venv уже собран, проверка ==='
  $NSPY -c "import numpy,torch,gsplat; from nerfstudio.models.splatfacto import SplatfactoModel; print('numpy',numpy.__version__,'| torch',torch.__version__,'| gsplat',gsplat.__version__,'| cuda',torch.cuda.is_available(),'| splatfacto OK')"
  exit 0
fi

pip install -q uv
uv python install 3.10
uv venv --python 3.10 /content/nsenv

uv pip install --python $NSPY "torch==2.1.2" "torchvision==0.16.2" --index-url https://download.pytorch.org/whl/cu118
uv pip install --python $NSPY "setuptools<70" wheel
uv pip install --python $NSPY "gsplat==1.4.0+pt21cu118" --index-url https://docs.gsplat.studio/whl/pt21cu118 --extra-index-url https://pypi.org/simple --index-strategy unsafe-best-match
uv pip install --python $NSPY "nerfstudio==1.1.5" open3d trimesh
uv pip install --python $NSPY "numpy==1.26.4"

echo '=== проверка ==='
$NSPY -c "import numpy,torch,gsplat; from gsplat.cuda._backend import _C; from nerfstudio.models.splatfacto import SplatfactoModel; print('numpy',numpy.__version__,'| torch',torch.__version__,'| gsplat',gsplat.__version__,'| cuda',torch.cuda.is_available(),'| splatfacto OK')"
/content/nsenv/bin/ns-train --help >/dev/null 2>&1 && echo 'ns-train: OK' || echo 'ns-train: FAIL'


In [1]:
%%bash
set -e
NSPY=/content/nsenv/bin/python

if [ -x /content/nsenv/bin/ns-train ]; then
  echo '=== venv уже собран, проверка ==='
  $NSPY -c "import numpy,torch,gsplat; from nerfstudio.models.splatfacto import SplatfactoModel; print('numpy',numpy.__version__,'| torch',torch.__version__,'| gsplat',gsplat.__version__,'| cuda',torch.cuda.is_available(),'| splatfacto OK')"
  exit 0
fi

pip install -q uv
uv python install 3.10
uv venv --python 3.10 /content/nsenv

uv pip install --python $NSPY "torch==2.1.2" "torchvision==0.16.2" --index-url https://download.pytorch.org/whl/cu118
uv pip install --python $NSPY "setuptools<70" wheel
uv pip install --python $NSPY "gsplat==1.4.0+pt21cu118" --index-url https://docs.gsplat.studio/whl/pt21cu118 --extra-index-url https://pypi.org/simple --index-strategy unsafe-best-match
uv pip install --python $NSPY "nerfstudio==1.1.5" open3d trimesh
uv pip install --python $NSPY "numpy==1.26.4"

echo '=== проверка ==='
$NSPY -c "import numpy,torch,gsplat; from gsplat.cuda._backend import _C; from nerfstudio.models.splatfacto import SplatfactoModel; print('numpy',numpy.__version__,'| torch',torch.__version__,'| gsplat',gsplat.__version__,'| cuda',torch.cuda.is_available(),'| splatfacto OK')"
/content/nsenv/bin/ns-train --help >/dev/null 2>&1 && echo 'ns-train: OK' || echo 'ns-train: FAIL'


In [2]:
!pip install -q -U transformers
import torch, numpy as np
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, Sam2Model, Sam2Processor

dev = "cuda"
OBJECT_PROMPT = "a truck."
GD, SAM = "IDEA-Research/grounding-dino-base", "facebook/sam2-hiera-large"
gd_proc = AutoProcessor.from_pretrained(GD)
gd_model = AutoModelForZeroShotObjectDetection.from_pretrained(GD).to(dev).eval()
sam_proc = Sam2Processor.from_pretrained(SAM)
sam_model = Sam2Model.from_pretrained(SAM).to(dev).eval()
print("модели ок (SAM2)")

def gdino_boxes(img, text=OBJECT_PROMPT, thr=0.3):
    inp = gd_proc(images=img, text=text, return_tensors="pt").to(dev)
    with torch.no_grad():
        out = gd_model(**inp)
    tgt = [img.size[::-1]]
    try:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    except TypeError:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, box_threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    return r["boxes"].cpu().numpy(), r["scores"].cpu().numpy()

_pp = getattr(sam_proc, "post_process_masks", None) or sam_proc.image_processor.post_process_masks

def sam_union(img, boxes):
    inp = sam_proc(img, input_boxes=[[list(map(float, b)) for b in boxes]], return_tensors="pt").to(dev)
    with torch.no_grad():
        out = sam_model(**inp, multimask_output=True)
    masks = _pp(out.pred_masks.cpu(), inp["original_sizes"].cpu())[0]
    m = masks.numpy()
    iou = out.iou_scores.cpu().numpy()[0]
    sel = np.stack([m[i, iou[i].argmax()] for i in range(m.shape[0])])
    return sel.any(0)


In [3]:
import os, subprocess

ABO_ITEM = "B07F2X8K62"
ABO_SPIN_ID = "f8022078"
ABO_SPIN_PREFIX = ABO_SPIN_ID[:2]
ABO_N_FRAMES = 72
D_ABO = "/content/abo_chair"
os.makedirs(f"{D_ABO}/raw", exist_ok=True)

base = "https://amazon-berkeley-objects.s3.amazonaws.com"
for az in range(ABO_N_FRAMES):
    fn = f"{ABO_SPIN_ID}_{az:02d}.jpg"
    url = f"{base}/spins/original/{ABO_SPIN_PREFIX}/{ABO_SPIN_ID}/{fn}"
    out = f"{D_ABO}/raw/{fn}"
    if not os.path.exists(out):
        subprocess.run(["curl", "-s", "-L", url, "-o", out], check=True)
print("фото скачано:", len(os.listdir(f"{D_ABO}/raw")))

cad_dir = f"{D_ABO}/cad"
os.makedirs(cad_dir, exist_ok=True)
cad_url = f"{base}/3dmodels/original/{ABO_ITEM[-1]}/{ABO_ITEM}.glb"
subprocess.run(["curl", "-s", "-L", cad_url, "-o", f"{cad_dir}/{ABO_ITEM}.glb"], check=True)
print("CAD скачан:", os.listdir(cad_dir))


In [4]:
import shutil
import numpy as np

def make_subset(n, raw_dir=f"{D_ABO}/raw", out_name=None):
    out_name = out_name or f"raw_N{n}"
    out_dir = f"{D_ABO}/{out_name}"
    if os.path.isdir(out_dir):
        shutil.rmtree(out_dir)
    os.makedirs(out_dir)
    names = sorted(os.listdir(raw_dir))
    idx = sorted(set(np.linspace(0, len(names) - 1, n).round().astype(int).tolist()))
    for i in idx:
        shutil.copy(f"{raw_dir}/{names[i]}", out_dir)
    print(out_name, "->", len(os.listdir(out_dir)), "фото")

make_subset(ABO_N_FRAMES, out_name="raw_dense")
make_subset(30)
make_subset(20)


In [5]:
import json

def pose_spherical(theta_deg, phi_deg, radius):
    def trans_t(t):
        return np.array([[1,0,0,0],[0,1,0,0],[0,0,1,t],[0,0,0,1]], dtype=float)
    def rot_phi(p):
        p = np.deg2rad(p)
        return np.array([[1,0,0,0],[0,np.cos(p),-np.sin(p),0],[0,np.sin(p),np.cos(p),0],[0,0,0,1]], dtype=float)
    def rot_theta(t):
        t = np.deg2rad(t)
        return np.array([[np.cos(t),0,-np.sin(t),0],[0,1,0,0],[np.sin(t),0,np.cos(t),0],[0,0,0,1]], dtype=float)
    c2w = trans_t(radius)
    c2w = rot_phi(phi_deg) @ c2w
    c2w = rot_theta(theta_deg) @ c2w
    c2w = np.array([[-1,0,0,0],[0,0,1,0],[0,1,0,0],[0,0,0,1]], dtype=float) @ c2w
    return c2w

def build_synthetic_dataset(subset_dir_name, radius=2.5, elevation_deg=-15.0, fov_deg=40.0):
    raw_dir = f"{D_ABO}/{subset_dir_name}"
    names = sorted(os.listdir(raw_dir))
    w, h = Image.open(f"{raw_dir}/{names[0]}").size
    out_dir = f"{D_ABO}/{subset_dir_name}_synth"
    img_out = f"{out_dir}/images"
    os.makedirs(img_out, exist_ok=True)
    fov_x = np.deg2rad(fov_deg)
    fl = 0.5 * w / np.tan(0.5 * fov_x)
    frames = []
    for n in names:
        az = int(n.split('_')[1].split('.')[0])
        theta = az * 5.0
        c2w = pose_spherical(theta, elevation_deg, radius)
        dst = f"{img_out}/{n}"
        if not os.path.exists(dst):
            shutil.copy(f"{raw_dir}/{n}", dst)
        frames.append({"file_path": f"images/{n}", "transform_matrix": c2w.tolist()})
    out = {
        "fl_x": fl, "fl_y": fl, "cx": w / 2, "cy": h / 2, "w": w, "h": h,
        "camera_model": "OPENCV", "k1": 0.0, "k2": 0.0, "p1": 0.0, "p2": 0.0,
        "frames": frames,
    }
    with open(f"{out_dir}/transforms.json", "w") as f:
        json.dump(out, f, indent=2)
    print(subset_dir_name, "-> поз:", len(frames), "->", out_dir)

for name in ["raw_dense", "raw_N30", "raw_N20"]:
    build_synthetic_dataset(name)


In [6]:
OBJECT_PROMPT_ABO = "a chair."

def make_masks(processed_dir, prompt=OBJECT_PROMPT_ABO):
    img_dir = f"{processed_dir}/images"
    mask_dir = f"{processed_dir}/masks"
    os.makedirs(mask_dir, exist_ok=True)
    names = sorted(os.listdir(img_dir))
    no_box = []
    for n in names:
        img = Image.open(f"{img_dir}/{n}").convert("RGB")
        W, H = img.size
        b, s = gdino_boxes(img, prompt, 0.3)
        if len(b) == 0:
            b, s = gdino_boxes(img, "furniture. object.", 0.22)
        keep = [bb for bb in b if 0.02 < ((bb[2]-bb[0])*(bb[3]-bb[1]))/(W*H) < 0.99]
        if not keep:
            no_box.append(n)
            mask = np.ones((H, W), bool)
        else:
            mask = sam_union(img, keep)
        Image.fromarray((mask*255).astype(np.uint8)).save(f"{mask_dir}/{os.path.splitext(n)[0]}.png")
    print(processed_dir, "-> масок:", len(os.listdir(mask_dir)), "| без бокса:", len(no_box), no_box[:5])

for name in ["raw_dense", "raw_N30", "raw_N20"]:
    make_masks(f"{D_ABO}/{name}_synth")


In [7]:
def add_mask_paths(out_name):
    d = f"{D_ABO}/{out_name}_synth"
    tf = f"{d}/transforms.json"
    data = json.load(open(tf))
    for fr in data["frames"]:
        fname = os.path.basename(fr["file_path"])
        stem = os.path.splitext(fname)[0]
        mask_file = f"masks/{stem}.png"
        if os.path.exists(f"{d}/{mask_file}"):
            fr["mask_path"] = mask_file
    json.dump(data, open(tf, "w"), indent=2)
    n_with_mask = sum(1 for fr in data["frames"] if "mask_path" in fr)
    print(out_name, "-> масок привязано:", n_with_mask, "/", len(data["frames"]))

for name in ["raw_dense", "raw_N30", "raw_N20"]:
    add_mask_paths(name)


In [8]:
import subprocess

def train_splatfacto(processed_dir, out_name, iters):
    cmd = (
        f"yes | TORCHDYNAMO_DISABLE=1 /content/nsenv/bin/ns-train splatfacto "
        f"--data {processed_dir} --max-num-iterations {iters} --steps-per-save 1000 "
        f"--vis tensorboard --viewer.quit-on-train-completion True "
        f"--output-dir {D_ABO}/outputs_{out_name} --pipeline.model.cull-alpha-thresh 0.1 "
        f"nerfstudio-data"
    )
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(out_name, "-> returncode", r.returncode)
    print(r.stdout[-1500:])
    if r.returncode:
        print(r.stderr[-1500:])

train_splatfacto(f"{D_ABO}/raw_dense_synth", "dense", iters=7000)
train_splatfacto(f"{D_ABO}/raw_N30_synth", "n30", iters=3000)
train_splatfacto(f"{D_ABO}/raw_N20_synth", "n20", iters=3000)


In [ ]:
import glob

def export_and_mesh(out_name):
    cfgs = glob.glob(f"{D_ABO}/outputs_{out_name}/*/splatfacto/*/config.yml")
    if not cfgs:
        print(out_name, "-> config.yml не найден")
        return
    cfg = sorted(cfgs)[-1]
    export_dir = f"{D_ABO}/exports_{out_name}"
    subprocess.run(f"/content/nsenv/bin/ns-export gaussian-splat --load-config {cfg} --output-dir {export_dir}", shell=True)

    script = f'''
import numpy as np
import open3d as o3d
import trimesh

pcd = o3d.io.read_point_cloud("{export_dir}/splat.ply")
pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
nn = o3d.geometry.KDTreeFlann(pcd)
step = max(1, len(pcd.points) // 2000)
d = []
for i in range(0, len(pcd.points), step):
    _, idx, dist2 = nn.search_knn_vector_3d(pcd.points[i], 2)
    d.append(np.sqrt(dist2[1]))
eps = 5 * float(np.median(d))
labels = np.array(pcd.cluster_dbscan(eps=eps, min_points=10))
if labels.max() >= 0:
    biggest = np.bincount(labels[labels >= 0]).argmax()
    pcd = pcd.select_by_index(np.where(labels == biggest)[0])
pcd.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
pcd.orient_normals_consistent_tangent_plane(30)
mesh, dens = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=9)
dens = np.asarray(dens)
mesh.remove_vertices_by_mask(dens <= np.quantile(dens, 0.1))
mesh.remove_degenerate_triangles(); mesh.remove_duplicated_vertices()
mesh.remove_duplicated_triangles(); mesh.remove_non_manifold_edges()
mesh.compute_triangle_normals()
out = "{export_dir}/chair_object.stl"
o3d.io.write_triangle_mesh(out, mesh)
m = trimesh.load(out)
m.fill_holes(); m.fix_normals(); m.process(validate=True)
m.export(out)
print("STL:", out, "| Ecken:", len(m.vertices), "| Dreiecke:", len(m.faces))
print("watertight:", m.is_watertight)
'''
    open(f"/content/mesh_abo_{out_name}.py", "w").write(script)
    r = subprocess.run(["/content/nsenv/bin/python", f"/content/mesh_abo_{out_name}.py"], capture_output=True, text=True)
    print(r.stdout)
    if r.returncode:
        print(r.stderr[-2000:])

for name in ["dense", "n30", "n20"]:
    export_and_mesh(name)


In [ ]:
from google.colab import files
files.download('/content/abo_chair/exports_dense/chair_object.stl')


In [1]:
import os
print("abo_chair:", os.path.exists('/content/abo_chair'))


In [2]:
%%bash
set -e
NSPY=/content/nsenv/bin/python

if [ -x /content/nsenv/bin/ns-train ]; then
  echo '=== venv уже собран, проверка ==='
  $NSPY -c "import numpy,torch,gsplat; from nerfstudio.models.splatfacto import SplatfactoModel; print('numpy',numpy.__version__,'| torch',torch.__version__,'| gsplat',gsplat.__version__,'| cuda',torch.cuda.is_available(),'| splatfacto OK')"
  exit 0
fi

pip install -q uv
uv python install 3.10
uv venv --python 3.10 /content/nsenv

uv pip install --python $NSPY "torch==2.1.2" "torchvision==0.16.2" --index-url https://download.pytorch.org/whl/cu118
uv pip install --python $NSPY "setuptools<70" wheel
uv pip install --python $NSPY "gsplat==1.4.0+pt21cu118" --index-url https://docs.gsplat.studio/whl/pt21cu118 --extra-index-url https://pypi.org/simple --index-strategy unsafe-best-match
uv pip install --python $NSPY "nerfstudio==1.1.5" open3d trimesh
uv pip install --python $NSPY "numpy==1.26.4"

echo '=== проверка ==='
$NSPY -c "import numpy,torch,gsplat; from gsplat.cuda._backend import _C; from nerfstudio.models.splatfacto import SplatfactoModel; print('numpy',numpy.__version__,'| torch',torch.__version__,'| gsplat',gsplat.__version__,'| cuda',torch.cuda.is_available(),'| splatfacto OK')"
/content/nsenv/bin/ns-train --help >/dev/null 2>&1 && echo 'ns-train: OK' || echo 'ns-train: FAIL'


In [3]:
!pip install -q -U transformers
import torch, numpy as np
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, Sam2Model, Sam2Processor

dev = "cuda"
OBJECT_PROMPT = "a truck."
GD, SAM = "IDEA-Research/grounding-dino-base", "facebook/sam2-hiera-large"
gd_proc = AutoProcessor.from_pretrained(GD)
gd_model = AutoModelForZeroShotObjectDetection.from_pretrained(GD).to(dev).eval()
sam_proc = Sam2Processor.from_pretrained(SAM)
sam_model = Sam2Model.from_pretrained(SAM).to(dev).eval()
print("модели ок (SAM2)")

def gdino_boxes(img, text=OBJECT_PROMPT, thr=0.3):
    inp = gd_proc(images=img, text=text, return_tensors="pt").to(dev)
    with torch.no_grad():
        out = gd_model(**inp)
    tgt = [img.size[::-1]]
    try:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    except TypeError:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, box_threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    return r["boxes"].cpu().numpy(), r["scores"].cpu().numpy()

_pp = getattr(sam_proc, "post_process_masks", None) or sam_proc.image_processor.post_process_masks

def sam_union(img, boxes):
    inp = sam_proc(img, input_boxes=[[list(map(float, b)) for b in boxes]], return_tensors="pt").to(dev)
    with torch.no_grad():
        out = sam_model(**inp, multimask_output=True)
    masks = _pp(out.pred_masks.cpu(), inp["original_sizes"].cpu())[0]
    m = masks.numpy()
    iou = out.iou_scores.cpu().numpy()[0]
    sel = np.stack([m[i, iou[i].argmax()] for i in range(m.shape[0])])
    return sel.any(0)


In [4]:
import os, subprocess

ABO_ITEM = "B07F2X8K62"
ABO_SPIN_ID = "f8022078"
ABO_SPIN_PREFIX = ABO_SPIN_ID[:2]
ABO_N_FRAMES = 72
D_ABO = "/content/abo_chair"
os.makedirs(f"{D_ABO}/raw", exist_ok=True)

base = "https://amazon-berkeley-objects.s3.amazonaws.com"
for az in range(ABO_N_FRAMES):
    fn = f"{ABO_SPIN_ID}_{az:02d}.jpg"
    url = f"{base}/spins/original/{ABO_SPIN_PREFIX}/{ABO_SPIN_ID}/{fn}"
    out = f"{D_ABO}/raw/{fn}"
    if not os.path.exists(out):
        subprocess.run(["curl", "-s", "-L", url, "-o", out], check=True)
print("фото скачано:", len(os.listdir(f"{D_ABO}/raw")))

cad_dir = f"{D_ABO}/cad"
os.makedirs(cad_dir, exist_ok=True)
cad_url = f"{base}/3dmodels/original/{ABO_ITEM[-1]}/{ABO_ITEM}.glb"
subprocess.run(["curl", "-s", "-L", cad_url, "-o", f"{cad_dir}/{ABO_ITEM}.glb"], check=True)
print("CAD скачан:", os.listdir(cad_dir))


In [5]:
import shutil
import numpy as np

def make_subset(n, raw_dir=f"{D_ABO}/raw", out_name=None):
    out_name = out_name or f"raw_N{n}"
    out_dir = f"{D_ABO}/{out_name}"
    if os.path.isdir(out_dir):
        shutil.rmtree(out_dir)
    os.makedirs(out_dir)
    names = sorted(os.listdir(raw_dir))
    idx = sorted(set(np.linspace(0, len(names) - 1, n).round().astype(int).tolist()))
    for i in idx:
        shutil.copy(f"{raw_dir}/{names[i]}", out_dir)
    print(out_name, "->", len(os.listdir(out_dir)), "фото")

make_subset(ABO_N_FRAMES, out_name="raw_dense")
make_subset(30)
make_subset(20)


In [6]:
import json

def pose_spherical(theta_deg, phi_deg, radius):
    def trans_t(t):
        return np.array([[1,0,0,0],[0,1,0,0],[0,0,1,t],[0,0,0,1]], dtype=float)
    def rot_phi(p):
        p = np.deg2rad(p)
        return np.array([[1,0,0,0],[0,np.cos(p),-np.sin(p),0],[0,np.sin(p),np.cos(p),0],[0,0,0,1]], dtype=float)
    def rot_theta(t):
        t = np.deg2rad(t)
        return np.array([[np.cos(t),0,-np.sin(t),0],[0,1,0,0],[np.sin(t),0,np.cos(t),0],[0,0,0,1]], dtype=float)
    c2w = trans_t(radius)
    c2w = rot_phi(phi_deg) @ c2w
    c2w = rot_theta(theta_deg) @ c2w
    c2w = np.array([[-1,0,0,0],[0,0,1,0],[0,1,0,0],[0,0,0,1]], dtype=float) @ c2w
    return c2w

def build_synthetic_dataset(subset_dir_name, radius=2.5, elevation_deg=-15.0, fov_deg=40.0):
    raw_dir = f"{D_ABO}/{subset_dir_name}"
    names = sorted(os.listdir(raw_dir))
    w, h = Image.open(f"{raw_dir}/{names[0]}").size
    out_dir = f"{D_ABO}/{subset_dir_name}_synth"
    img_out = f"{out_dir}/images"
    os.makedirs(img_out, exist_ok=True)
    fov_x = np.deg2rad(fov_deg)
    fl = 0.5 * w / np.tan(0.5 * fov_x)
    frames = []
    for n in names:
        az = int(n.split('_')[1].split('.')[0])
        theta = az * 5.0
        c2w = pose_spherical(theta, elevation_deg, radius)
        dst = f"{img_out}/{n}"
        if not os.path.exists(dst):
            shutil.copy(f"{raw_dir}/{n}", dst)
        frames.append({"file_path": f"images/{n}", "transform_matrix": c2w.tolist()})
    out = {
        "fl_x": fl, "fl_y": fl, "cx": w / 2, "cy": h / 2, "w": w, "h": h,
        "camera_model": "OPENCV", "k1": 0.0, "k2": 0.0, "p1": 0.0, "p2": 0.0,
        "frames": frames,
    }
    with open(f"{out_dir}/transforms.json", "w") as f:
        json.dump(out, f, indent=2)
    print(subset_dir_name, "-> поз:", len(frames), "->", out_dir)

for name in ["raw_dense", "raw_N30", "raw_N20"]:
    build_synthetic_dataset(name)


In [7]:
OBJECT_PROMPT_ABO = "a chair."

def make_masks(processed_dir, prompt=OBJECT_PROMPT_ABO):
    img_dir = f"{processed_dir}/images"
    mask_dir = f"{processed_dir}/masks"
    os.makedirs(mask_dir, exist_ok=True)
    names = sorted(os.listdir(img_dir))
    no_box = []
    for n in names:
        img = Image.open(f"{img_dir}/{n}").convert("RGB")
        W, H = img.size
        b, s = gdino_boxes(img, prompt, 0.3)
        if len(b) == 0:
            b, s = gdino_boxes(img, "furniture. object.", 0.22)
        keep = [bb for bb in b if 0.02 < ((bb[2]-bb[0])*(bb[3]-bb[1]))/(W*H) < 0.99]
        if not keep:
            no_box.append(n)
            mask = np.ones((H, W), bool)
        else:
            mask = sam_union(img, keep)
        Image.fromarray((mask*255).astype(np.uint8)).save(f"{mask_dir}/{os.path.splitext(n)[0]}.png")
    print(processed_dir, "-> масок:", len(os.listdir(mask_dir)), "| без бокса:", len(no_box), no_box[:5])

for name in ["raw_dense", "raw_N30", "raw_N20"]:
    make_masks(f"{D_ABO}/{name}_synth")


In [8]:
def add_mask_paths(out_name):
    d = f"{D_ABO}/{out_name}_synth"
    tf = f"{d}/transforms.json"
    data = json.load(open(tf))
    for fr in data["frames"]:
        fname = os.path.basename(fr["file_path"])
        stem = os.path.splitext(fname)[0]
        mask_file = f"masks/{stem}.png"
        if os.path.exists(f"{d}/{mask_file}"):
            fr["mask_path"] = mask_file
    json.dump(data, open(tf, "w"), indent=2)
    n_with_mask = sum(1 for fr in data["frames"] if "mask_path" in fr)
    print(out_name, "-> масок привязано:", n_with_mask, "/", len(data["frames"]))

for name in ["raw_dense", "raw_N30", "raw_N20"]:
    add_mask_paths(name)


In [9]:
import subprocess

def train_splatfacto(processed_dir, out_name, iters):
    cmd = (
        f"yes | TORCHDYNAMO_DISABLE=1 /content/nsenv/bin/ns-train splatfacto "
        f"--data {processed_dir} --max-num-iterations {iters} --steps-per-save 1000 "
        f"--vis tensorboard --viewer.quit-on-train-completion True "
        f"--output-dir {D_ABO}/outputs_{out_name} --pipeline.model.cull-alpha-thresh 0.1 "
        f"nerfstudio-data"
    )
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(out_name, "-> returncode", r.returncode)
    print(r.stdout[-1500:])
    if r.returncode:
        print(r.stderr[-1500:])

train_splatfacto(f"{D_ABO}/raw_dense_synth", "dense", iters=7000)
train_splatfacto(f"{D_ABO}/raw_N30_synth", "n30", iters=3000)
train_splatfacto(f"{D_ABO}/raw_N20_synth", "n20", iters=3000)


In [10]:
import glob

def export_and_mesh(out_name):
    cfgs = glob.glob(f"{D_ABO}/outputs_{out_name}/*/splatfacto/*/config.yml")
    if not cfgs:
        print(out_name, "-> config.yml не найден")
        return
    cfg = sorted(cfgs)[-1]
    export_dir = f"{D_ABO}/exports_{out_name}"
    subprocess.run(f"/content/nsenv/bin/ns-export gaussian-splat --load-config {cfg} --output-dir {export_dir}", shell=True)

    script = f'''
import numpy as np
import open3d as o3d
import trimesh

pcd = o3d.io.read_point_cloud("{export_dir}/splat.ply")
pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
nn = o3d.geometry.KDTreeFlann(pcd)
step = max(1, len(pcd.points) // 2000)
d = []
for i in range(0, len(pcd.points), step):
    _, idx, dist2 = nn.search_knn_vector_3d(pcd.points[i], 2)
    d.append(np.sqrt(dist2[1]))
eps = 5 * float(np.median(d))
labels = np.array(pcd.cluster_dbscan(eps=eps, min_points=10))
if labels.max() >= 0:
    biggest = np.bincount(labels[labels >= 0]).argmax()
    pcd = pcd.select_by_index(np.where(labels == biggest)[0])
pcd.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
pcd.orient_normals_consistent_tangent_plane(30)
mesh, dens = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=9)
dens = np.asarray(dens)
mesh.remove_vertices_by_mask(dens <= np.quantile(dens, 0.1))
mesh.remove_degenerate_triangles(); mesh.remove_duplicated_vertices()
mesh.remove_duplicated_triangles(); mesh.remove_non_manifold_edges()
mesh.compute_triangle_normals()
out = "{export_dir}/chair_object.stl"
o3d.io.write_triangle_mesh(out, mesh)
m = trimesh.load(out)
m.fill_holes(); m.fix_normals(); m.process(validate=True)
m.export(out)
print("STL:", out, "| Ecken:", len(m.vertices), "| Dreiecke:", len(m.faces))
print("watertight:", m.is_watertight)
'''
    open(f"/content/mesh_abo_{out_name}.py", "w").write(script)
    r = subprocess.run(["/content/nsenv/bin/python", f"/content/mesh_abo_{out_name}.py"], capture_output=True, text=True)
    print(r.stdout)
    if r.returncode:
        print(r.stderr[-2000:])

for name in ["dense", "n30"]:
    export_and_mesh(name)


In [11]:
from google.colab import files
files.download('/content/abo_chair/exports_dense/chair_object.stl')


In [12]:
import os
print(os.path.exists('/content/abo_chair/exports_dense/chair_object.stl'))


In [13]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy('/content/abo_chair/exports_dense/chair_object.stl', '/content/drive/MyDrive/chair_object_dense.stl')
shutil.copy('/content/abo_chair/exports_n30/chair_object.stl', '/content/drive/MyDrive/chair_object_n30.stl')
print("скопировано на Drive")


In [1]:
import subprocess

def train_splatfacto(processed_dir, out_name, iters):
    cmd = (
        f"yes | TORCHDYNAMO_DISABLE=1 /content/nsenv/bin/ns-train splatfacto "
        f"--data {processed_dir} --max-num-iterations {iters} --steps-per-save 1000 "
        f"--vis tensorboard --viewer.quit-on-train-completion True "
        f"--output-dir {D_ABO}/outputs_{out_name} --pipeline.model.cull-alpha-thresh 0.1 "
        f"--pipeline.model.camera-optimizer.mode SO3xR3 "
        f"nerfstudio-data"
    )
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(out_name, "-> returncode", r.returncode)
    print(r.stdout[-1500:])
    if r.returncode:
        print(r.stderr[-1500:])

train_splatfacto(f"{D_ABO}/raw_dense_synth", "dense", iters=7000)
train_splatfacto(f"{D_ABO}/raw_N30_synth", "n30", iters=3000)
train_splatfacto(f"{D_ABO}/raw_N20_synth", "n20", iters=3000)


In [2]:
import os
print(os.path.exists('/content/abo_chair'))
print(os.listdir('/content/abo_chair') if os.path.exists('/content/abo_chair') else "нет — runtime сброшен")


In [3]:
%%bash
set -e
NSPY=/content/nsenv/bin/python

if [ -x /content/nsenv/bin/ns-train ]; then
  echo '=== venv уже собран, проверка ==='
  $NSPY -c "import numpy,torch,gsplat; from nerfstudio.models.splatfacto import SplatfactoModel; print('numpy',numpy.__version__,'| torch',torch.__version__,'| gsplat',gsplat.__version__,'| cuda',torch.cuda.is_available(),'| splatfacto OK')"
  exit 0
fi

pip install -q uv
uv python install 3.10
uv venv --python 3.10 /content/nsenv

uv pip install --python $NSPY "torch==2.1.2" "torchvision==0.16.2" --index-url https://download.pytorch.org/whl/cu118
uv pip install --python $NSPY "setuptools<70" wheel
uv pip install --python $NSPY "gsplat==1.4.0+pt21cu118" --index-url https://docs.gsplat.studio/whl/pt21cu118 --extra-index-url https://pypi.org/simple --index-strategy unsafe-best-match
uv pip install --python $NSPY "nerfstudio==1.1.5" open3d trimesh
uv pip install --python $NSPY "numpy==1.26.4"

echo '=== проверка ==='
$NSPY -c "import numpy,torch,gsplat; from gsplat.cuda._backend import _C; from nerfstudio.models.splatfacto import SplatfactoModel; print('numpy',numpy.__version__,'| torch',torch.__version__,'| gsplat',gsplat.__version__,'| cuda',torch.cuda.is_available(),'| splatfacto OK')"
/content/nsenv/bin/ns-train --help >/dev/null 2>&1 && echo 'ns-train: OK' || echo 'ns-train: FAIL'


In [4]:
import os, subprocess

ABO_ITEM = "B07F2X8K62"
ABO_SPIN_ID = "f8022078"
ABO_SPIN_PREFIX = ABO_SPIN_ID[:2]
ABO_N_FRAMES = 72
D_ABO = "/content/abo_chair"
os.makedirs(f"{D_ABO}/raw", exist_ok=True)

base = "https://amazon-berkeley-objects.s3.amazonaws.com"
for az in range(ABO_N_FRAMES):
    fn = f"{ABO_SPIN_ID}_{az:02d}.jpg"
    url = f"{base}/spins/original/{ABO_SPIN_PREFIX}/{ABO_SPIN_ID}/{fn}"
    out = f"{D_ABO}/raw/{fn}"
    if not os.path.exists(out):
        r = subprocess.run(["curl", "-s", "-L", "-o", out, url], capture_output=True, text=True)
        if r.returncode:
            print(fn, "FAIL", r.stderr[-200:])

print("скачано:", len(os.listdir(f"{D_ABO}/raw")), "/", ABO_N_FRAMES)


In [5]:
import shutil
import numpy as np

def make_subset(n, raw_dir=f"{D_ABO}/raw", out_name=None):
    out_name = out_name or f"raw_N{n}"
    out_dir = f"{D_ABO}/{out_name}"
    if os.path.isdir(out_dir):
        shutil.rmtree(out_dir)
    os.makedirs(out_dir)
    names = sorted(os.listdir(raw_dir))
    idx = sorted(set(np.linspace(0, len(names) - 1, n).round().astype(int).tolist()))
    for i in idx:
        shutil.copy(f"{raw_dir}/{names[i]}", out_dir)
    print(out_name, "->", len(os.listdir(out_dir)), "фото")

make_subset(ABO_N_FRAMES, out_name="raw_dense")
make_subset(30)
make_subset(20)


In [6]:
import json

def pose_spherical(theta_deg, phi_deg, radius):
    def trans_t(t):
        return np.array([[1,0,0,0],[0,1,0,0],[0,0,1,t],[0,0,0,1]], dtype=float)
    def rot_phi(p):
        p = np.deg2rad(p)
        return np.array([[1,0,0,0],[0,np.cos(p),-np.sin(p),0],[0,np.sin(p),np.cos(p),0],[0,0,0,1]], dtype=float)
    def rot_theta(t):
        t = np.deg2rad(t)
        return np.array([[np.cos(t),0,-np.sin(t),0],[0,1,0,0],[np.sin(t),0,np.cos(t),0],[0,0,0,1]], dtype=float)
    c2w = trans_t(radius)
    c2w = rot_phi(phi_deg) @ c2w
    c2w = rot_theta(theta_deg) @ c2w
    c2w = np.array([[-1,0,0,0],[0,0,1,0],[0,1,0,0],[0,0,0,1]], dtype=float) @ c2w
    return c2w

def build_synthetic_dataset(subset_dir_name, radius=2.5, elevation_deg=-15.0, fov_deg=40.0):
    raw_dir = f"{D_ABO}/{subset_dir_name}"
    names = sorted(os.listdir(raw_dir))
    w, h = Image.open(f"{raw_dir}/{names[0]}").size
    out_dir = f"{D_ABO}/{subset_dir_name}_synth"
    img_out = f"{out_dir}/images"
    os.makedirs(img_out, exist_ok=True)
    fov_x = np.deg2rad(fov_deg)
    fl = 0.5 * w / np.tan(0.5 * fov_x)
    frames = []
    for n in names:
        az = int(n.split('_')[1].split('.')[0])
        theta = az * 5.0
        c2w = pose_spherical(theta, elevation_deg, radius)
        dst = f"{img_out}/{n}"
        if not os.path.exists(dst):
            shutil.copy(f"{raw_dir}/{n}", dst)
        frames.append({"file_path": f"images/{n}", "transform_matrix": c2w.tolist()})
    out = {
        "fl_x": fl, "fl_y": fl, "cx": w / 2, "cy": h / 2, "w": w, "h": h,
        "camera_model": "OPENCV", "k1": 0.0, "k2": 0.0, "p1": 0.0, "p2": 0.0,
        "frames": frames,
    }
    with open(f"{out_dir}/transforms.json", "w") as f:
        json.dump(out, f, indent=2)
    print(subset_dir_name, "-> поз:", len(frames), "->", out_dir)

for name in ["raw_dense", "raw_N30", "raw_N20"]:
    build_synthetic_dataset(name)


In [7]:
from PIL import Image


In [8]:
for name in ["raw_dense", "raw_N30", "raw_N20"]:
    build_synthetic_dataset(name)


In [9]:
!pip install -q -U transformers
import torch, numpy as np
from PIL import Image
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, Sam2Model, Sam2Processor

dev = "cuda"
GD, SAM = "IDEA-Research/grounding-dino-base", "facebook/sam2-hiera-large"
gd_proc = AutoProcessor.from_pretrained(GD)
gd_model = AutoModelForZeroShotObjectDetection.from_pretrained(GD).to(dev).eval()
sam_proc = Sam2Processor.from_pretrained(SAM)
sam_model = Sam2Model.from_pretrained(SAM).to(dev).eval()
print("модели ок (SAM2)")

def gdino_boxes(img, text, thr=0.3):
    inp = gd_proc(images=img, text=text, return_tensors="pt").to(dev)
    with torch.no_grad():
        out = gd_model(**inp)
    tgt = [img.size[::-1]]
    try:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    except TypeError:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, box_threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    return r["boxes"].cpu().numpy(), r["scores"].cpu().numpy()

_pp = getattr(sam_proc, "post_process_masks", None) or sam_proc.image_processor.post_process_masks

def sam_union(img, boxes):
    inp = sam_proc(img, input_boxes=[[list(map(float, b)) for b in boxes]], return_tensors="pt").to(dev)
    with torch.no_grad():
        out = sam_model(**inp, multimask_output=True)
    masks = _pp(out.pred_masks.cpu(), inp["original_sizes"].cpu())[0]
    m = masks.numpy()
    iou = out.iou_scores.cpu().numpy()[0]
    sel = np.stack([m[i, iou[i].argmax()] for i in range(m.shape[0])])
    return sel.any(0)


In [10]:
OBJECT_PROMPT_ABO = "a chair."

def make_masks(processed_dir, prompt=OBJECT_PROMPT_ABO):
    img_dir = f"{processed_dir}/images"
    mask_dir = f"{processed_dir}/masks"
    os.makedirs(mask_dir, exist_ok=True)
    names = sorted(os.listdir(img_dir))
    no_box = []
    for n in names:
        img = Image.open(f"{img_dir}/{n}").convert("RGB")
        W, H = img.size
        b, s = gdino_boxes(img, prompt, 0.3)
        if len(b) == 0:
            b, s = gdino_boxes(img, "furniture. object.", 0.22)
        keep = [bb for bb in b if 0.02 < ((bb[2]-bb[0])*(bb[3]-bb[1]))/(W*H) < 0.99]
        if not keep:
            no_box.append(n)
            mask = np.ones((H, W), bool)
        else:
            mask = sam_union(img, keep)
        Image.fromarray((mask*255).astype(np.uint8)).save(f"{mask_dir}/{os.path.splitext(n)[0]}.png")
    print(processed_dir, "-> масок:", len(os.listdir(mask_dir)), "| без бокса:", len(no_box), no_box[:5])

for name in ["raw_dense", "raw_N30", "raw_N20"]:
    make_masks(f"{D_ABO}/{name}_synth")


In [11]:
def add_mask_paths(out_name):
    d = f"{D_ABO}/{out_name}_synth"
    tf = f"{d}/transforms.json"
    data = json.load(open(tf))
    for fr in data["frames"]:
        fname = os.path.basename(fr["file_path"])
        stem = os.path.splitext(fname)[0]
        mask_file = f"masks/{stem}.png"
        if os.path.exists(f"{d}/{mask_file}"):
            fr["mask_path"] = mask_file
    json.dump(data, open(tf, "w"), indent=2)
    n_with_mask = sum(1 for fr in data["frames"] if "mask_path" in fr)
    print(out_name, "-> масок привязано:", n_with_mask, "/", len(data["frames"]))

for name in ["raw_dense", "raw_N30", "raw_N20"]:
    add_mask_paths(name)


In [12]:
import subprocess

def train_splatfacto(processed_dir, out_name, iters):
    cmd = (
        f"yes | TORCHDYNAMO_DISABLE=1 /content/nsenv/bin/ns-train splatfacto "
        f"--data {processed_dir} --max-num-iterations {iters} --steps-per-save 1000 "
        f"--vis tensorboard --viewer.quit-on-train-completion True "
        f"--output-dir {D_ABO}/outputs_{out_name} --pipeline.model.cull-alpha-thresh 0.1 "
        f"--pipeline.model.camera-optimizer.mode SO3xR3 "
        f"nerfstudio-data"
    )
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(out_name, "-> returncode", r.returncode)
    print(r.stdout[-1500:])
    if r.returncode:
        print(r.stderr[-1500:])

train_splatfacto(f"{D_ABO}/raw_dense_synth", "dense", iters=7000)
train_splatfacto(f"{D_ABO}/raw_N30_synth", "n30", iters=3000)
train_splatfacto(f"{D_ABO}/raw_N20_synth", "n20", iters=3000)


In [1]:
import os
print("runtime жив:", os.path.exists('/content/abo_chair'))
if os.path.exists('/content/abo_chair'):
    import glob
    print(glob.glob('/content/abo_chair/outputs_*/*/splatfacto/*/config.yml'))


In [1]:
%%bash
set -e
NSPY=/content/nsenv/bin/python

if [ -x /content/nsenv/bin/ns-train ]; then
  echo '=== venv уже собран, проверка ==='
  $NSPY -c "import numpy,torch,gsplat; from nerfstudio.models.splatfacto import SplatfactoModel; print('numpy',numpy.__version__,'| torch',torch.__version__,'| gsplat',gsplat.__version__,'| cuda',torch.cuda.is_available(),'| splatfacto OK')"
  exit 0
fi

pip install -q uv
uv python install 3.10
uv venv --python 3.10 /content/nsenv

uv pip install --python $NSPY "torch==2.1.2" "torchvision==0.16.2" --index-url https://download.pytorch.org/whl/cu118
uv pip install --python $NSPY "setuptools<70" wheel
uv pip install --python $NSPY "gsplat==1.4.0+pt21cu118" --index-url https://docs.gsplat.studio/whl/pt21cu118 --extra-index-url https://pypi.org/simple --index-strategy unsafe-best-match
uv pip install --python $NSPY "nerfstudio==1.1.5" open3d trimesh
uv pip install --python $NSPY "numpy==1.26.4"

echo '=== проверка ==='
$NSPY -c "import numpy,torch,gsplat; from gsplat.cuda._backend import _C; from nerfstudio.models.splatfacto import SplatfactoModel; print('numpy',numpy.__version__,'| torch',torch.__version__,'| gsplat',gsplat.__version__,'| cuda',torch.cuda.is_available(),'| splatfacto OK')"
/content/nsenv/bin/ns-train --help >/dev/null 2>&1 && echo 'ns-train: OK' || echo 'ns-train: FAIL'


In [2]:
import os, subprocess

ABO_ITEM = "B07F2X8K62"
ABO_SPIN_ID = "f8022078"
ABO_SPIN_PREFIX = ABO_SPIN_ID[:2]
ABO_N_FRAMES = 72
D_ABO = "/content/abo_chair"
os.makedirs(f"{D_ABO}/raw", exist_ok=True)

base = "https://amazon-berkeley-objects.s3.amazonaws.com"
for az in range(ABO_N_FRAMES):
    fn = f"{ABO_SPIN_ID}_{az:02d}.jpg"
    url = f"{base}/spins/original/{ABO_SPIN_PREFIX}/{ABO_SPIN_ID}/{fn}"
    out = f"{D_ABO}/raw/{fn}"
    if not os.path.exists(out):
        subprocess.run(["curl", "-s", "-L", url, "-o", out], check=True)
print("фото скачано:", len(os.listdir(f"{D_ABO}/raw")))


In [3]:
import shutil
import numpy as np

def make_subset(n, raw_dir=f"{D_ABO}/raw", out_name=None):
    out_name = out_name or f"raw_N{n}"
    out_dir = f"{D_ABO}/{out_name}"
    if os.path.isdir(out_dir):
        shutil.rmtree(out_dir)
    os.makedirs(out_dir)
    names = sorted(os.listdir(raw_dir))
    idx = sorted(set(np.linspace(0, len(names) - 1, n).round().astype(int).tolist()))
    for i in idx:
        shutil.copy(f"{raw_dir}/{names[i]}", out_dir)
    print(out_name, "->", len(os.listdir(out_dir)), "фото")

make_subset(ABO_N_FRAMES, out_name="raw_dense")
make_subset(30)
make_subset(20)


In [4]:
from PIL import Image
import json

def pose_spherical(theta_deg, phi_deg, radius):
    def trans_t(t):
        return np.array([[1,0,0,0],[0,1,0,0],[0,0,1,t],[0,0,0,1]], dtype=float)
    def rot_phi(p):
        p = np.deg2rad(p)
        return np.array([[1,0,0,0],[0,np.cos(p),-np.sin(p),0],[0,np.sin(p),np.cos(p),0],[0,0,0,1]], dtype=float)
    def rot_theta(t):
        t = np.deg2rad(t)
        return np.array([[np.cos(t),0,-np.sin(t),0],[0,1,0,0],[np.sin(t),0,np.cos(t),0],[0,0,0,1]], dtype=float)
    c2w = trans_t(radius)
    c2w = rot_phi(phi_deg) @ c2w
    c2w = rot_theta(theta_deg) @ c2w
    c2w = np.array([[-1,0,0,0],[0,0,1,0],[0,1,0,0],[0,0,0,1]], dtype=float) @ c2w
    return c2w

def build_synthetic_dataset(subset_dir_name, radius=2.5, elevation_deg=-15.0, fov_deg=40.0):
    raw_dir = f"{D_ABO}/{subset_dir_name}"
    names = sorted(os.listdir(raw_dir))
    w, h = Image.open(f"{raw_dir}/{names[0]}").size
    out_dir = f"{D_ABO}/{subset_dir_name}_synth"
    img_out = f"{out_dir}/images"
    os.makedirs(img_out, exist_ok=True)
    fov_x = np.deg2rad(fov_deg)
    fl = 0.5 * w / np.tan(0.5 * fov_x)
    frames = []
    for n in names:
        az = int(n.split('_')[1].split('.')[0])
        theta = az * 5.0
        c2w = pose_spherical(theta, elevation_deg, radius)
        dst = f"{img_out}/{n}"
        if not os.path.exists(dst):
            shutil.copy(f"{raw_dir}/{n}", dst)
        frames.append({"file_path": f"images/{n}", "transform_matrix": c2w.tolist()})
    out = {
        "fl_x": fl, "fl_y": fl, "cx": w / 2, "cy": h / 2, "w": w, "h": h,
        "camera_model": "OPENCV", "k1": 0.0, "k2": 0.0, "p1": 0.0, "p2": 0.0,
        "frames": frames,
    }
    with open(f"{out_dir}/transforms.json", "w") as f:
        json.dump(out, f, indent=2)
    print(subset_dir_name, "-> поз:", len(frames), "->", out_dir)

for name in ["raw_dense", "raw_N30", "raw_N20"]:
    build_synthetic_dataset(name)


In [5]:
!pip install -q -U transformers
import torch, numpy as np
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, Sam2Model, Sam2Processor

dev = "cuda"
GD, SAM = "IDEA-Research/grounding-dino-base", "facebook/sam2-hiera-large"
gd_proc = AutoProcessor.from_pretrained(GD)
gd_model = AutoModelForZeroShotObjectDetection.from_pretrained(GD).to(dev).eval()
sam_proc = Sam2Processor.from_pretrained(SAM)
sam_model = Sam2Model.from_pretrained(SAM).to(dev).eval()
print("модели ок (SAM2)")

def gdino_boxes(img, text, thr=0.3):
    inp = gd_proc(images=img, text=text, return_tensors="pt").to(dev)
    with torch.no_grad():
        out = gd_model(**inp)
    tgt = [img.size[::-1]]
    try:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    except TypeError:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, box_threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    return r["boxes"].cpu().numpy(), r["scores"].cpu().numpy()

_pp = getattr(sam_proc, "post_process_masks", None) or sam_proc.image_processor.post_process_masks

def sam_union(img, boxes):
    inp = sam_proc(img, input_boxes=[[list(map(float, b)) for b in boxes]], return_tensors="pt").to(dev)
    with torch.no_grad():
        out = sam_model(**inp, multimask_output=True)
    masks = _pp(out.pred_masks.cpu(), inp["original_sizes"].cpu())[0]
    m = masks.numpy()
    iou = out.iou_scores.cpu().numpy()[0]
    sel = np.stack([m[i, iou[i].argmax()] for i in range(m.shape[0])])
    return sel.any(0)


In [6]:
OBJECT_PROMPT_ABO = "a chair."

def make_masks(processed_dir, prompt=OBJECT_PROMPT_ABO):
    img_dir = f"{processed_dir}/images"
    mask_dir = f"{processed_dir}/masks"
    os.makedirs(mask_dir, exist_ok=True)
    names = sorted(os.listdir(img_dir))
    no_box = []
    for n in names:
        img = Image.open(f"{img_dir}/{n}").convert("RGB")
        W, H = img.size
        b, s = gdino_boxes(img, prompt, 0.3)
        if len(b) == 0:
            b, s = gdino_boxes(img, "furniture. object.", 0.22)
        keep = [bb for bb in b if 0.02 < ((bb[2]-bb[0])*(bb[3]-bb[1]))/(W*H) < 0.99]
        if not keep:
            no_box.append(n)
            mask = np.ones((H, W), bool)
        else:
            mask = sam_union(img, keep)
        Image.fromarray((mask*255).astype(np.uint8)).save(f"{mask_dir}/{os.path.splitext(n)[0]}.png")
    print(processed_dir, "-> масок:", len(os.listdir(mask_dir)), "| без бокса:", len(no_box), no_box[:5])

for name in ["raw_dense", "raw_N30", "raw_N20"]:
    make_masks(f"{D_ABO}/{name}_synth")


In [7]:
def add_mask_paths(out_name):
    d = f"{D_ABO}/{out_name}_synth"
    tf = f"{d}/transforms.json"
    data = json.load(open(tf))
    for fr in data["frames"]:
        fname = os.path.basename(fr["file_path"])
        stem = os.path.splitext(fname)[0]
        mask_file = f"masks/{stem}.png"
        if os.path.exists(f"{d}/{mask_file}"):
            fr["mask_path"] = mask_file
    json.dump(data, open(tf, "w"), indent=2)
    n_with_mask = sum(1 for fr in data["frames"] if "mask_path" in fr)
    print(out_name, "-> масок привязано:", n_with_mask, "/", len(data["frames"]))

for name in ["raw_dense", "raw_N30", "raw_N20"]:
    add_mask_paths(name)


In [8]:
import subprocess

def train_splatfacto(processed_dir, out_name, iters):
    cmd = (
        f"yes | TORCHDYNAMO_DISABLE=1 /content/nsenv/bin/ns-train splatfacto "
        f"--data {processed_dir} --max-num-iterations {iters} --steps-per-save 1000 "
        f"--vis tensorboard --viewer.quit-on-train-completion True "
        f"--output-dir {D_ABO}/outputs_{out_name} --pipeline.model.cull-alpha-thresh 0.1 "
        f"--pipeline.model.camera-optimizer.mode SO3xR3 "
        f"nerfstudio-data"
    )
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(out_name, "-> returncode", r.returncode)
    print(r.stdout[-1500:])
    if r.returncode:
        print(r.stderr[-1500:])

train_splatfacto(f"{D_ABO}/raw_dense_synth", "dense", iters=7000)
train_splatfacto(f"{D_ABO}/raw_N30_synth", "n30", iters=3000)
train_splatfacto(f"{D_ABO}/raw_N20_synth", "n20", iters=3000)


In [9]:
import glob

def export_and_mesh(out_name):
    cfgs = glob.glob(f"{D_ABO}/outputs_{out_name}/*/splatfacto/*/config.yml")
    if not cfgs:
        print(out_name, "-> config.yml не найден")
        return
    cfg = sorted(cfgs)[-1]
    export_dir = f"{D_ABO}/exports_{out_name}"
    subprocess.run(f"/content/nsenv/bin/ns-export gaussian-splat --load-config {cfg} --output-dir {export_dir}", shell=True)

    script = f'''
import numpy as np
import open3d as o3d
import trimesh

pcd = o3d.io.read_point_cloud("{export_dir}/splat.ply")
pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
nn = o3d.geometry.KDTreeFlann(pcd)
step = max(1, len(pcd.points) // 2000)
d = []
for i in range(0, len(pcd.points), step):
    _, idx, dist2 = nn.search_knn_vector_3d(pcd.points[i], 2)
    d.append(np.sqrt(dist2[1]))
eps = 5 * float(np.median(d))
labels = np.array(pcd.cluster_dbscan(eps=eps, min_points=10))
if labels.max() >= 0:
    biggest = np.bincount(labels[labels >= 0]).argmax()
    pcd = pcd.select_by_index(np.where(labels == biggest)[0])
pcd.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
pcd.orient_normals_consistent_tangent_plane(30)
mesh, dens = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=9)
dens = np.asarray(dens)
mesh.remove_vertices_by_mask(dens <= np.quantile(dens, 0.1))
mesh.remove_degenerate_triangles(); mesh.remove_duplicated_vertices()
mesh.remove_duplicated_triangles(); mesh.remove_non_manifold_edges()
mesh.compute_triangle_normals()
out = "{export_dir}/chair_object.stl"
o3d.io.write_triangle_mesh(out, mesh)
m = trimesh.load(out)
m.fill_holes(); m.fix_normals(); m.process(validate=True)
m.export(out)
bbox = m.bounds[1] - m.bounds[0]
print("STL:", out, "| Ecken:", len(m.vertices), "| Dreiecke:", len(m.faces))
print("bbox (LxWxH):", bbox)
print("watertight:", m.is_watertight)
'''
    open(f"/content/mesh_abo_{out_name}.py", "w").write(script)
    r = subprocess.run(["/content/nsenv/bin/python", f"/content/mesh_abo_{out_name}.py"], capture_output=True, text=True)
    print(r.stdout)
    if r.returncode:
        print(r.stderr[-2000:])

for name in ["dense", "n30", "n20"]:
    export_and_mesh(name)


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
CACHE = '/content/drive/MyDrive/nerf_cache/content_cache.tar.gz'
if not os.path.exists('/content/nsenv/bin/ns-train') and os.path.exists(CACHE):
    print('Восстанавливаю venv из кэша...')
    os.system(f'tar -xzf {CACHE} -C / content/nsenv')
print('venv:', os.path.exists('/content/nsenv/bin/ns-train'))

os.system('apt-get -qq install -y colmap > /dev/null')
print('colmap:', shutil.which('colmap'))

RAW = '/content/figurine/raw'
os.makedirs(RAW, exist_ok=True)
os.system(f'unzip -q -o /content/drive/MyDrive/figurine/figurine_raw.zip -d {RAW}')
print('фото:', len(os.listdir(RAW)))


In [2]:
!ls -la /content/drive/MyDrive/nerf_cache/ 2>&1 | head
!ls -la /content/drive/MyDrive/figurine/ 2>&1 | head
!find /content/drive/MyDrive -maxdepth 2 -iname "*figurine*" 2>/dev/null


In [3]:
!ls /content/drive/MyDrive | head -30


In [4]:
!find /content/drive/MyDrive -maxdepth 3 \( -iname "*figurine*" -o -iname "content_cache*" \) 2>/dev/null


In [5]:
%%bash
set -e
NSPY=/content/nsenv/bin/python
pip install -q uv
uv python install 3.10
uv venv --python 3.10 /content/nsenv
uv pip install --python $NSPY "torch==2.1.2" "torchvision==0.16.2" --index-url https://download.pytorch.org/whl/cu118
uv pip install --python $NSPY "setuptools<70" wheel
uv pip install --python $NSPY "gsplat==1.4.0+pt21cu118" --index-url https://docs.gsplat.studio/whl/pt21cu118 --extra-index-url https://pypi.org/simple --index-strategy unsafe-best-match
uv pip install --python $NSPY "nerfstudio==1.1.5" open3d trimesh
uv pip install --python $NSPY "numpy==1.26.4"
$NSPY -c "import numpy,torch,gsplat; from nerfstudio.models.splatfacto import SplatfactoModel; print('numpy',numpy.__version__,'| torch',torch.__version__,'| cuda',torch.cuda.is_available(),'| splatfacto OK')"


In [6]:
import os
RAW = '/content/figurine/raw'
os.makedirs(RAW, exist_ok=True)
os.system(f'unzip -q -o /content/drive/MyDrive/figurine/figurine_raw.zip -d {RAW}')
print('фото:', len(os.listdir(RAW)))

!rm -rf /content/figurine/processed
!QT_QPA_PLATFORM=offscreen /content/nsenv/bin/ns-process-data images \
  --data /content/figurine/raw \
  --output-dir /content/figurine/processed \
  --matching-method exhaustive \
  --no-gpu 2>&1 | tail -15


In [7]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)
!find /content/drive/MyDrive -iname "*figurine*" 2>/dev/null


In [8]:
!find /content/drive/MyDrive -iname "*figurine*" 2>/dev/null


In [9]:
import os
RAW = '/content/figurine/raw'
os.makedirs(RAW, exist_ok=True)
os.system(f'unzip -q -o /content/drive/MyDrive/figurine/figurine_raw.zip -d {RAW}')
print('фото:', len(os.listdir(RAW)))

!rm -rf /content/figurine/processed
!QT_QPA_PLATFORM=offscreen /content/nsenv/bin/ns-process-data images \
  --data /content/figurine/raw \
  --output-dir /content/figurine/processed \
  --matching-method exhaustive \
  --no-gpu 2>&1 | tail -15


In [10]:
!find /content/drive/MyDrive -maxdepth 3 -iname "pictures" -type d
!ls /content/drive/MyDrive/pictures | head -5; ls /content/drive/MyDrive/pictures | wc -l


In [11]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import glob, os
from PIL import Image, ImageOps
src = [p for p in glob.glob('/content/drive/MyDrive/**/pictures', recursive=True) if os.path.isdir(p)]
print('найдено:', src)

RAW = '/content/figurine/raw'
os.makedirs(RAW, exist_ok=True)
n = 0
for f in sorted(glob.glob(src[0] + '/*.JPG') + glob.glob(src[0] + '/*.jpg')):
    im = Image.open(f)
    if max(im.size) < 1000:      # битые превью 240x320
        continue
    im = ImageOps.exif_transpose(im).convert('RGB')
    im.thumbnail((1600, 1600), Image.LANCZOS)
    im.save(f'{RAW}/{os.path.splitext(os.path.basename(f))[0]}.jpg', quality=95)
    n += 1
print('фото готово:', n, im.size)


In [12]:
!rm -rf /content/figurine/processed
!QT_QPA_PLATFORM=offscreen /content/nsenv/bin/ns-process-data images \
  --data /content/figurine/raw \
  --output-dir /content/figurine/processed \
  --matching-method exhaustive \
  --no-gpu 2>&1 | tail -15


In [13]:
!pip install -q -U transformers
import os, torch, numpy as np
from PIL import Image
from tqdm import tqdm
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, Sam2Model, Sam2Processor

dev = "cuda"
OBJECT_PROMPT = "a figurine."
GD, SAM = "IDEA-Research/grounding-dino-base", "facebook/sam2-hiera-large"
gd_proc = AutoProcessor.from_pretrained(GD)
gd_model = AutoModelForZeroShotObjectDetection.from_pretrained(GD).to(dev).eval()
sam_proc = Sam2Processor.from_pretrained(SAM)
sam_model = Sam2Model.from_pretrained(SAM).to(dev).eval()

def gdino_boxes(img, text, thr=0.3):
    inp = gd_proc(images=img, text=text, return_tensors="pt").to(dev)
    with torch.no_grad():
        out = gd_model(**inp)
    tgt = [img.size[::-1]]
    try:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    except TypeError:
        r = gd_proc.post_process_grounded_object_detection(out, inp.input_ids, box_threshold=thr, text_threshold=0.25, target_sizes=tgt)[0]
    return r["boxes"].cpu().numpy(), r["scores"].cpu().numpy()

_pp = getattr(sam_proc, "post_process_masks", None) or sam_proc.image_processor.post_process_masks
def sam_union(img, boxes):
    inp = sam_proc(img, input_boxes=[[list(map(float, b)) for b in boxes]], return_tensors="pt").to(dev)
    with torch.no_grad():
        out = sam_model(**inp, multimask_output=True)
    m = _pp(out.pred_masks.cpu(), inp["original_sizes"].cpu())[0].numpy()
    iou = out.iou_scores.cpu().numpy()[0]
    return np.stack([m[i, iou[i].argmax()] for i in range(m.shape[0])]).any(0)

D = '/content/figurine/processed'
IMG_DIR, MASK_DIR = f'{D}/images', f'{D}/masks'
os.makedirs(MASK_DIR, exist_ok=True)
names = sorted(os.listdir(IMG_DIR))
no_box, previews = [], []
for i, n in enumerate(tqdm(names)):
    img = Image.open(f"{IMG_DIR}/{n}").convert("RGB")
    W, H = img.size
    b, s = gdino_boxes(img, OBJECT_PROMPT, 0.3)
    if len(b) == 0:
        b, s = gdino_boxes(img, "toy. statue. robot.", 0.22)
    keep = [bb for bb in b if 0.02 < ((bb[2]-bb[0])*(bb[3]-bb[1]))/(W*H) < 0.99]
    if not keep:
        no_box.append(n); mask = np.ones((H, W), bool)
    else:
        mask = sam_union(img, keep)
    Image.fromarray((mask*255).astype(np.uint8)).save(f"{MASK_DIR}/{os.path.splitext(n)[0]}.png")
    if i % 5 == 0:
        ov = np.array(img); ov[~mask] = (ov[~mask]*0.2).astype(np.uint8)
        previews.append(Image.fromarray(ov).resize((240, 320)))
print("масок:", len(os.listdir(MASK_DIR)), "| без бокса:", len(no_box), no_box)

sheet = Image.new('RGB', (240*len(previews), 320))
for k, p in enumerate(previews): sheet.paste(p, (240*k, 0))
display(sheet)


In [14]:
%cd /content/figurine
!yes | TORCHDYNAMO_DISABLE=1 /content/nsenv/bin/ns-train splatfacto \
  --data /content/figurine/processed \
  --max-num-iterations 7000 --steps-per-save 2000 \
  --vis tensorboard --viewer.quit-on-train-completion True \
  --output-dir outputs_fig --pipeline.model.cull-alpha-thresh 0.1 \
  colmap --images-path images --colmap-path colmap/sparse/0 --masks-path masks --downscale-factor 1 2>&1 | tail -20


In [15]:
import glob, os, subprocess
WORK = '/content/figurine'
cfg = sorted(glob.glob(f'{WORK}/outputs_fig/*/splatfacto/*/config.yml'))[-1]
EXPORT = f'{WORK}/export'
subprocess.run(f'/content/nsenv/bin/ns-export gaussian-splat --load-config {cfg} --output-dir {EXPORT}', shell=True)

script = f'''
import numpy as np, open3d as o3d, trimesh
pcd = o3d.io.read_point_cloud("{EXPORT}/splat.ply")
print("Gaussians:", len(pcd.points))
pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
nn = o3d.geometry.KDTreeFlann(pcd)
step = max(1, len(pcd.points) // 2000)
d = [np.sqrt(nn.search_knn_vector_3d(pcd.points[i], 2)[2][1]) for i in range(0, len(pcd.points), step)]
eps = 5 * float(np.median(d))
labels = np.array(pcd.cluster_dbscan(eps=eps, min_points=10))
if labels.max() >= 0:
    pcd = pcd.select_by_index(np.where(labels == np.bincount(labels[labels >= 0]).argmax())[0])
print("nach DBSCAN:", len(pcd.points))
pcd.estimate_normals(o3d.geometry.KDTreeSearchParamKNN(30))
pcd.orient_normals_consistent_tangent_plane(30)
mesh, dens = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=9)
dens = np.asarray(dens)
mesh.remove_vertices_by_mask(dens <= np.quantile(dens, 0.1))
mesh.remove_degenerate_triangles(); mesh.remove_duplicated_vertices()
mesh.remove_duplicated_triangles(); mesh.remove_non_manifold_edges()
mesh.compute_triangle_normals()
out = "{EXPORT}/object.stl"
o3d.io.write_triangle_mesh(out, mesh)
m = trimesh.load(out)
m.fill_holes(); m.fix_normals(); m.process(validate=True)
m.export(out)
print("STL:", out, "| Ecken:", len(m.vertices), "| Dreiecke:", len(m.faces))
print("Bounding Box:", m.extents.round(3), "| watertight:", m.is_watertight)
'''
open('/content/mesh_object.py', 'w').write(script)
r = subprocess.run(['/content/nsenv/bin/python', '/content/mesh_object.py'], capture_output=True, text=True)
print(r.stdout); print(r.stderr[-2000:] if r.returncode else '')


In [16]:
import shutil, trimesh, numpy as np, matplotlib.pyplot as plt
OUT = '/content/drive/MyDrive/Applied_Robotics/results'
os.makedirs(OUT, exist_ok=True)
shutil.copy(f'{EXPORT}/object.stl', f'{OUT}/figurine_object.stl')
shutil.copy(f'{EXPORT}/splat.ply', f'{OUT}/figurine_splat.ply')
print(os.listdir(OUT))

m = trimesh.load(f'{EXPORT}/object.stl')
p = m.vertices[np.random.choice(len(m.vertices), 30000, replace=False)]
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for a, (i, j), t in zip(ax, [(0, 1), (0, 2), (1, 2)], ['XY', 'XZ', 'YZ']):
    a.scatter(p[:, i], p[:, j], s=0.2, c=p[:, 3 - i - j], cmap='viridis'); a.set_title(t); a.set_aspect('equal')
plt.show()


In [17]:
!pip install -q trimesh
import shutil, trimesh, numpy as np, matplotlib.pyplot as plt
OUT = '/content/drive/MyDrive/Applied_Robotics/results'
os.makedirs(OUT, exist_ok=True)
shutil.copy(f'{EXPORT}/object.stl', f'{OUT}/figurine_object.stl')
shutil.copy(f'{EXPORT}/splat.ply', f'{OUT}/figurine_splat.ply')
print(os.listdir(OUT))

m = trimesh.load(f'{EXPORT}/object.stl')
p = m.vertices[np.random.choice(len(m.vertices), 30000, replace=False)]
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for a, (i, j), t in zip(ax, [(0, 1), (0, 2), (1, 2)], ['XY', 'XZ', 'YZ']):
    a.scatter(p[:, i], p[:, j], s=0.2, c=p[:, 3 - i - j], cmap='viridis'); a.set_title(t); a.set_aspect('equal')
plt.show()


In [18]:
script = f'''
import numpy as np, torch, open3d as o3d, trimesh
from pathlib import Path
from PIL import Image
from nerfstudio.utils.eval_utils import eval_setup

_, pipe, _, _ = eval_setup(Path("{cfg}"), test_mode="test")
model, ds = pipe.model, pipe.datamanager.train_dataset
X = model.means.detach().cpu().numpy()
op = torch.sigmoid(model.opacities).detach().cpu().numpy().ravel()
print("Gaussians:", len(X))

cams = ds.cameras
c2w = cams.camera_to_worlds.cpu().numpy()
fx, fy = cams.fx.cpu().numpy().ravel(), cams.fy.cpu().numpy().ravel()
cx, cy = cams.cx.cpu().numpy().ravel(), cams.cy.cpu().numpy().ravel()
masks = ds._dataparser_outputs.mask_filenames
inside = np.zeros(len(X)); seen = np.zeros(len(X))
for k in range(len(c2w)):
    M = np.array(Image.open(masks[k])) > 127
    H, W = M.shape[:2]
    R, t = c2w[k][:, :3], c2w[k][:, 3]
    Pc = (X - t) @ R                      # OpenGL-Kamera: z zeigt nach hinten
    z = -Pc[:, 2]
    u = fx[k] * Pc[:, 0] / z + cx[k]
    v = -fy[k] * Pc[:, 1] / z + cy[k]
    ok = (z > 0) & (u >= 0) & (u < W) & (v >= 0) & (v < H)
    seen += ok
    ui, vi = u[ok].astype(int), v[ok].astype(int)
    inside[np.where(ok)[0]] += M[vi, ui]
frac = np.where(seen > 0, inside / np.maximum(seen, 1), 0)
keep = (frac >= 0.9) & (seen >= 10) & (op > 0.1)
print("nach Masken-Carving:", keep.sum())

pcd = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(X[keep]))
pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
o3d.io.write_point_cloud("{EXPORT}/carved.ply", pcd)
pcd.estimate_normals(o3d.geometry.KDTreeSearchParamKNN(30))
pcd.orient_normals_consistent_tangent_plane(30)
mesh, dens = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=9)
dens = np.asarray(dens)
mesh.remove_vertices_by_mask(dens <= np.quantile(dens, 0.1))
mesh.remove_degenerate_triangles(); mesh.remove_duplicated_vertices()
mesh.remove_duplicated_triangles(); mesh.remove_non_manifold_edges()
mesh.compute_triangle_normals()
out = "{EXPORT}/object_carved.stl"
o3d.io.write_triangle_mesh(out, mesh)
m = trimesh.load(out); m.fill_holes(); m.fix_normals(); m.process(validate=True); m.export(out)
print("STL:", out, "| Ecken:", len(m.vertices), "| Dreiecke:", len(m.faces))
print("Bounding Box:", m.extents.round(3), "| watertight:", m.is_watertight)
'''
open('/content/carve.py', 'w').write(script)
r = subprocess.run(['/content/nsenv/bin/python', '/content/carve.py'], capture_output=True, text=True)
print(r.stdout[-3000:]); print(r.stderr[-3000:] if r.returncode else '')

m = trimesh.load(f'{EXPORT}/object_carved.stl')
p = m.vertices[np.random.choice(len(m.vertices), min(30000, len(m.vertices)), replace=False)]
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for a, (i, j), t in zip(ax, [(0, 1), (0, 2), (1, 2)], ['XY', 'XZ', 'YZ']):
    a.scatter(p[:, i], p[:, j], s=0.2, c=p[:, 3 - i - j], cmap='viridis'); a.set_title(t); a.set_aspect('equal')
plt.show()


In [19]:
shutil.copy(f'{EXPORT}/object_carved.stl', f'{OUT}/figurine_object_carved.stl')
print(os.listdir(OUT))


In [ ]:
!find /content/drive/MyDrive -maxdepth 3 -iname "pictures" -type d
!ls /content/drive/MyDrive/pictures | head -5; ls /content/drive/MyDrive/pictures | wc -l


In [20]:
script = f'''
import numpy as np, torch, trimesh
from pathlib import Path
from PIL import Image
from skimage.measure import marching_cubes
from nerfstudio.utils.eval_utils import eval_setup

_, pipe, _, _ = eval_setup(Path("{cfg}"), test_mode="test")
ds = pipe.datamanager.train_dataset
cams = ds.cameras
c2w = cams.camera_to_worlds.cuda()
fx, fy, cx, cy = [getattr(cams, a).cuda().ravel() for a in ("fx", "fy", "cx", "cy")]
masks = [torch.from_numpy(np.array(Image.open(f)) > 127).cuda() for f in ds._dataparser_outputs.mask_filenames]

pts = np.asarray(__import__("open3d").io.read_point_cloud("{EXPORT}/carved.ply").points)
lo, hi = pts.min(0), pts.max(0); pad = 0.15 * (hi - lo); lo, hi = lo - pad, hi + pad
N = 256
ax = [torch.linspace(float(lo[i]), float(hi[i]), N, device="cuda") for i in range(3)]
G = torch.stack(torch.meshgrid(*ax, indexing="ij"), -1).reshape(-1, 3)
inside = torch.zeros(len(G), device="cuda"); seen = torch.zeros(len(G), device="cuda")
for k in range(len(masks)):
    M = masks[k]; H, W = M.shape[:2]
    R, t = c2w[k][:, :3], c2w[k][:, 3]
    Pc = (G - t) @ R
    z = -Pc[:, 2]
    u = fx[k] * Pc[:, 0] / z + cx[k]
    v = -fy[k] * Pc[:, 1] / z + cy[k]
    ok = (z > 0) & (u >= 0) & (u < W) & (v >= 0) & (v < H)
    seen += ok
    inside[ok] += M[v[ok].long(), u[ok].long()].float()
occ = ((inside / seen.clamp(min=1) >= 0.95) & (seen >= 10)).reshape(N, N, N).cpu().numpy()
print("Voxel belegt:", occ.sum())

vol = np.pad(occ.astype(np.float32), 1)
step = (hi - lo) / (N - 1)
verts, faces, _, _ = marching_cubes(vol, 0.5, spacing=tuple(step))
verts += lo - step
m = trimesh.Trimesh(verts, faces)
m = max(m.split(only_watertight=False), key=lambda c: len(c.faces))
trimesh.smoothing.filter_taubin(m, iterations=20)
m.fix_normals()
m.export("{EXPORT}/object_hull.stl")
print("STL:", len(m.vertices), "Ecken |", len(m.faces), "Dreiecke | Box:", m.extents.round(3), "| watertight:", m.is_watertight)
'''
open('/content/hull.py', 'w').write(script)
r = subprocess.run(['/content/nsenv/bin/python', '/content/hull.py'], capture_output=True, text=True)
print(r.stdout[-2000:]); print(r.stderr[-2000:] if r.returncode else '')
shutil.copy(f'{EXPORT}/object_hull.stl', f'{OUT}/figurine_object_hull.stl')
